# Consumer Financial Complaint Classification
### INFO 4360 – Complex Data Analytics · Final project

**Team members:** Rebecca Quaye and Ketty Odero
**Analysis path:** Path A – predictive classification
**Data:** 5,000 CFPB consumer complaints, August 2025 – July 2026, 11 financial product categories

## The problem

A complaint-routing team receives complaints by email, chat and phone with **no product label**. Someone has to read each one and send it to the right one of 11 product teams; a wrong choice delays the answer and makes the consumer explain again. CFPB complaints come with the product the consumer chose, so they let us train and test a model that reads the complaint text and predicts the product.

**Research question:** can the complaint narrative alone route a complaint to the right product team, and which approach does it best: classic text models, embedding similarity, or Claude?

## How this notebook is organised

| Section | Content |
|---|---|
| 0 – 2 | Setup, data exploration and text mining (Phase 2) |
| 3 | Predictive modeling: seven approaches plus a baseline, one train/test split, model choice by cross-validation, then product-level, robustness, routing and subgroup checks |
| 4 | Claude: prompts, error analysis, the team's hand check, cost and trade-offs |
| 5 | Final report: key findings, business implications, ethics and privacy, AI usage log, limitations |

Every Claude answer is saved in `Data/` (and Rebecca's label-only descriptions in `outputs/rebecca_modelling/`), so the notebook re-runs without an API key and without new charges.

## 0. Setup
### 0.1 Imports

In [ ]:
# Notebook display, file handling, text patterns, and reproducible project paths
%matplotlib inline
import sys
import os
import re
import json
import time
from pathlib import Path
from collections import Counter

# Locate the repository root whether the notebook starts in Notebook/ or in the repository root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))           # makes the src/ helpers importable
DATA_DIR = PROJECT_ROOT / "Data"                    # data sample and saved Claude answers
OUTPUT_DIR = PROJECT_ROOT / "outputs"               # saved charts, tables and Rebecca's label-only descriptions
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Data handling and notebook tables
import numpy as np
import pandas as pd
from IPython.display import display

# Visualization: charts, heatmaps and word clouds
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Text processing for the word counts in sections 2.2 and 2.3 (NLTK)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download(["stopwords", "wordnet"], quiet=True)   # word lists for the cleaning step (skipped if already installed)

# Project helper functions (src/data_utils.py, src/preprocessing.py, src/text_analysis.py)
from src.data_utils import load_complaints, missing_values, duplicate_narratives, TEXT_COL, SHORT_NAMES
from src.preprocessing import (clean_text, drop_duplicate_narratives, top_terms,
                               distinctive_terms, product_word_matrix, STOPWORDS)
from src.text_analysis import (sentiment_scores, sentiment_label, positive_word_contributions,
                               FINANCE_WORDS)

# Modeling (section 3): TF-IDF, three classifiers, validation and metrics
from sklearn.base import clone                      # copy a model's settings without its fitted values
from sklearn.dummy import DummyClassifier           # majority-class baseline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline               # TF-IDF + classifier as one model
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import normalize         # unit-length vectors for cosine similarity
import spacy                                        # word vectors for model 4 (class 6, Exercise 3)
# (sentence-transformers for models 5 and 6 is imported in 3.5, so the rest runs without it)

# Claude API (sections 3 and 4); the key is read from .env and never printed
from concurrent.futures import ThreadPoolExecutor   # send several requests at the same time
import anthropic
from dotenv import load_dotenv

# One consistent chart style for every figure
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95,
              rc={"axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlesize": 12})
MAIN = sns.color_palette("deep")[0]      # one colour for every single-series chart
ACCENT = sns.color_palette("deep")[3]    # used only to highlight a key value
pd.set_option("display.max_colwidth", 200)   # show up to 200 characters of text in tables
print("Project root:", PROJECT_ROOT)

### 0.2 Load the data

In [ ]:
# Load the complaint sample. load_complaints reads the CSV, parses the dates and adds
# three helper columns: month, product_short (short product names) and word_count
DATA_PATH = DATA_DIR / "cfpb_complaints_sample_2025-08_to_2026-07.csv"
df = load_complaints(DATA_PATH)

## 1. Data exploration

### 1.1 Size and structure

A quick look at how many complaints we have, the period they cover, and the columns we will use.
Note: the 19 columns include 3 helper columns created by `load_complaints` (`month`, `product_short`, `word_count`).

In [ ]:
# Preview the first two rows with every column, to see what each field looks like
df.head(2)

In [ ]:
# --- Dataset size and scope ---
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")

# Date range covered by the sample (should run Aug 2025 to Jul 2026)
print(f"Complaints received: {df['Date received'].min():%b %d, %Y} to {df['Date received'].max():%b %d, %Y}")

# Number of target classes (products) and how many different companies appear
print(f"Product categories: {df['Product'].nunique()}   Companies: {df['Company'].nunique()}")

# --- Preview: first 5 complaints, showing the key columns only ---
df[["Date received", "product_short", "Issue", "word_count"]].head()

### 1.2 Missing values

In [ ]:
# Count missing values per column (only columns with at least one gap are listed)
missing_values(df)

The two columns we need, the narrative (input X) and the product (target Y), have **no missing values**, so all 5,000 complaints can be used and no rows are dropped. The gaps are in fields we will not use as model inputs: `Tags` (84% missing) is only filled for older Americans and servicemembers, and `Company public response` (69%) is optional. `Sub-issue` (28%) is blank for issues that have no sub-issues.

### 1.3 Target distribution: our sample vs. the full database

In the full CFPB database one category dominates. We sampled the same number of complaints per product so the model learns every category, not just credit reporting.

In [ ]:
# Count complaints by product (the target variable we will predict)
product_counts = df["Product"].value_counts()

product_counts

In [ ]:
# Product counts shown on the CFPB complaint search page (all complaints, Oct 2026)
database_counts = pd.Series({
    "Credit reporting": 6_341_482, "Debt collection": 317_439, "Credit card": 92_750,
    "Checking/savings": 85_931, "Money transfer": 38_217, "Mortgage": 32_110,
    "Vehicle loan": 24_617, "Student loan": 19_519, "Payday/personal loan": 17_907,
    "Prepaid card": 6_122, "Debt management": 4_243,
})
# Convert counts to % shares so the full database and our sample can be compared on one scale
database_share = database_counts / database_counts.sum() * 100
sample_share = df["product_short"].value_counts(normalize=True) * 100

order = database_share.sort_values().index      # same product order in both panels

shares = pd.DataFrame({"Full CFPB database": database_share,
                       "Our sample (5,000 complaints)": sample_share}).reindex(order[::-1])

# Two side-by-side bar charts: full database (left) vs. our sample (right)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
for ax, column in zip(axes, shares.columns):
    sns.barplot(x=shares[column], y=shares.index, color=MAIN, ax=ax)
    ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=8)   # write the % at the end of each bar
    ax.set(xlim=(0, 100), xlabel="Share of complaints (%)", ylabel="", title=column)

fig.suptitle("Credit reporting is 91% of all complaints; our sample gives each product an equal 9%",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
sns.despine(left=True)
plt.tight_layout()
plt.show()

- Our sample is **balanced by design**: 454 or 455 complaints for each of the 11 products (table above).
- The real complaint stream is extremely imbalanced: credit reporting is about 91%, and the six smallest products are each under 0.5%. A model trained on raw data could score 91% accuracy by always guessing "credit reporting" and still be useless for routing.
- Because the sample is balanced, **accuracy is a fair headline metric**, and **macro F1** (which weights every product equally) shows whether small products are handled well.
- For Phase 3: a model trained on balanced data will over-predict small products when used on real traffic. We will note this and consider adjusting for real-world frequencies when interpreting results.

### 1.4 Coverage over time

In [ ]:
# Number of complaints received in each month of the sample
per_month = df["month"].value_counts().sort_index()
print(per_month.to_string())

# Smallest count in the month x product table: if it is above 0, every product appears in every month
print(f"\nSmallest product-month cell: {pd.crosstab(df['month'], df['Product']).min().min()} complaints")

- Every month from August 2025 to May 2026 has 418 complaints; June (413) and July (407) are slightly lower, likely because fewer recent narratives were published before the CFPB stopped publishing narratives in August 2026. Every product appears in every month (the smallest product-month count is 37), so no single month drives the patterns we find.

### 1.5 Complaint Narrative Length

We examine the length of the consumer complaint narratives to understand how much the amount of text varies across complaints.

In [ ]:
# Number of words in each narrative (helper column created by load_complaints)
words = df["word_count"]

# Plain (linear) axis: show the chart up to the 99th percentile so the long tail
# doesn't squash the bars; the statistics below still use every complaint
cap = words.quantile(0.99)
shown = words[words <= cap]

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(shown, bins=40, color=MAIN, kde=True, ax=ax)   # bars = counts, line = smoothed shape
# Dashed line and label at the median
ax.axvline(words.median(), color=ACCENT, linewidth=2, linestyle="--")
ax.text(words.median() + cap * 0.01, ax.get_ylim()[1] * 0.93, f"Median: {words.median():.0f} words",
        color=ACCENT, fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))   # 1,000 instead of 1000
ax.set(xlim=(0, cap), xlabel="Words per narrative", ylabel="Number of complaints",
       title="Half of all narratives are 87 to 281 words; a long tail runs past 1,000")
# Note on the chart how many long narratives fall outside the plotted range
ax.text(0.99, 0.80, f"{(words > cap).sum()} narratives over {cap:,.0f} words not shown",
        transform=ax.transAxes, ha="right", fontsize=8, color="grey")
sns.despine()
plt.tight_layout()
plt.show()

# Summary statistics on ALL narratives, plus the share of very short and very long ones
print(words.describe().round(0).to_string())
print(f"Under 50 words: {(words < 50).mean():.0%}   Over 500 words: {(words > 500).mean():.0%}")

Complaint narrative lengths are right-skewed, with most complaints containing relatively short narratives and fewer complaints having very long narratives. The median narrative is 170 words, while the mean is higher at 225 words, and the longest runs to 5,058 words. This difference reflects a smaller number of very long complaints: 7% are over 500 words, while 12% are under 50 words.

For readability, the histogram displays narratives up to the 99th percentile. The full dataset, including longer narratives, is retained for the analysis. The variation in narrative length may be important when preprocessing the text and building the classification model.

### 1.6 Data quality: redactions and template letters

In [ ]:
# Share of narratives containing CFPB redactions (two or more X's in a row, e.g. XXXX)
has_redaction = df[TEXT_COL].str.contains(r"X{2,}")
print(f"Narratives with redacted text (XXXX): {has_redaction.mean():.0%}")

# Narratives whose text appears more than once (template letters)
dupes = duplicate_narratives(df)
extra_copies = int((dupes["copies"] - 1).sum())   # copies beyond the first one of each letter
print(f"Narratives that appear more than once: {len(dupes)}  ({extra_copies} extra copies)")
dupes.head(5)   # the five most repeated letters

- **82% of narratives contain CFPB redactions** (`XXXX` for names, account numbers and dates). These are meaningless to a model, so the cleaning step will remove them before counting words.
- **105 rows are exact copies** of 44 template letters, mostly credit reporting disputes citing FCRA sections. If copies land in both the training and test sets, the model is tested on text it has already seen and its score is inflated. We remove the extra copies before modeling, leaving 4,895 unique narratives.

### 1.7 Example narratives (one per product)

In [ ]:
# Display one sample complaint narrative per product (random_state fixes the sample so it is reproducible)
sample_complaints = (
    df[["Product", "Consumer complaint narrative"]]
    .groupby("Product")
    .sample(n=1, random_state=42)
    .sort_values("Product")
)

# Print the product and the first 500 characters of its narrative
for _, row in sample_complaints.iterrows():
    print("Product:", row["Product"])
    print("Narrative:", row["Consumer complaint narrative"][:500])
    print("-" * 80)

The sample complaints vary in length and writing style, but they contain language that provides clues about the financial product involved. Some name the product outright ("my mortgage account", "private student loan", "my vehicle", "Amex giftcard"), while others only point to it through context, such as a company or service name (PayPal for money transfer, Cash App Borrow for a payday loan). The mortgage example is itself about credit reporting of a late payment, so a single complaint can touch more than one product.

This supports the use of text classification for categorizing unlabeled complaints, but it also highlights a potential concern: explicit product words may make some complaints easier to classify. We will examine this further in the modeling phase as part of our leakage and sanity checks.

## 2. Text mining

### 2.1 Preparing the text

Before counting words we (1) keep one copy of each duplicate narrative, so a template letter sent 27 times is not counted 27 times, and (2) clean each narrative with `clean_text` from `src/preprocessing.py`: lower-case, and remove CFPB redactions (`XXXX`), dollar amounts, numbers and punctuation.

For the simple word counts in 2.2 and 2.3 we use a second, stricter cleaning (`clean_for_counts`): lower-case, keep letters only, drop English stopwords and redaction placeholders, and lemmatize ("payments" → "payment"), so that counts group word forms together. These counts run on all 5,000 narratives.

In [ ]:
# Remove duplicate template letters, then clean every narrative
text_df = drop_duplicate_narratives(df)                 # keep one copy of each template letter
text_df["clean"] = text_df[TEXT_COL].map(clean_text)   # cleaned text used for TF-IDF and sentiment checks
print(f"Unique narratives used for text mining: {len(text_df):,}")

# Before / after cleaning for one complaint
example = text_df.iloc[3]
print("\nBEFORE:", example[TEXT_COL][:300])
print("\nAFTER: ", example["clean"][:300])

In [ ]:
# Stricter cleaning for raw word counts (stopwords removed, lemmatized)
stop_words = set(stopwords.words("english"))   # common English words such as "the", "and", "is"
lemmatizer = WordNetLemmatizer()               # reduces words to their base form ("payments" -> "payment")

def clean_for_counts(text):
    # Convert text to lowercase
    text = text.lower()
    
    # Remove numbers and special characters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Split text into individual words
    words = text.split()

    # Remove stopwords and redaction placeholders
    words = [
    word for word in words
    if word not in stop_words and not re.fullmatch(r"x+", word)
    ]

    # Lemmatize words
    words = [lemmatizer.lemmatize(word) for word in words]
    
    # Join words back into text
    return " ".join(words)

# Apply stricter cleaning to unique narratives for word-frequency analysis
text_df["clean_narrative"] = text_df[TEXT_COL].apply(clean_for_counts)


### 2.2 Most common words and phrases across all complaints

We start with single words, then move to two-word phrases (bigrams), which carry more meaning.

In [ ]:
# Count the most common single words across all narratives

# Combine cleaned words from unique complaint narratives
all_words = " ".join(text_df["clean_narrative"]).split()
# Count how often each word appears
word_counts = Counter(all_words)   # Counter maps each word to the number of times it appears

# Get the 20 most common words
top_words = pd.DataFrame(
    word_counts.most_common(20),
    columns=["Word", "Frequency"]
)
# Plot the 20 most frequent words
plt.figure(figsize=(10, 7))

plt.barh(top_words["Word"], top_words["Frequency"])

plt.title("Top 20 Most Frequent Words in Consumer Complaints")
plt.xlabel("Word Frequency")
plt.ylabel("Words")

# Display the most frequent word at the top
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

### Interpretation

- **Account** is the most frequently used word, followed by **payment**, **credit**, and **loan**. These terms reflect common financial issues discussed in consumer complaints.
- Most of the frequently occurring words are general financial terms that appear across different product categories.
- Although individual words help us understand the main topics in complaints, they may not be enough to distinguish financial products. This is why we also examine bigrams and other text-mining techniques.

**Note:** This analysis uses 4,895 unique complaint narratives after removing duplicates to avoid counting repeated text multiple times.

In [ ]:
# Top 15 two-word phrases across all complaints
top_phrases = top_terms(text_df["clean"], n=15, ngram_range=(2, 2))   # ngram_range=(2, 2): two-word phrases only

# Horizontal bar chart, most frequent phrase at the top
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top_phrases, x="count", y="term", color=MAIN, ax=ax)
ax.bar_label(ax.containers[0], padding=3, fontsize=8)   # write the count at the end of each bar
ax.set(xlabel="Number of times the phrase appears", ylabel="",
       title="Common phrases are credit-reporting legal language and company names")
sns.despine(left=True)
plt.tight_layout()
plt.show()

- The two most frequent phrases are "credit reporting" (1,097) and "credit report" (894), and several others ("fair credit", "reporting act", "act fcra") come from **legal language about the Fair Credit Reporting Act**. Consumers in several products, not only credit reporting, quote this law, so these phrases alone will not separate products cleanly.
- **Company names** are among the most common phrases: "wells fargo" (679) and "cash app" (632) rank third and fourth, and "bank america" (359) also appears. A company usually points to a product (Cash App means money transfer), so the model may lean on them. That is useful for routing at one bank, but we will check in Phase 3 that the model is not relying on company names alone.
- "Identity theft" (603 mentions) is a major theme on its own.

### 2.3 The words that distinguish each product

**Raw counts first.** The most frequent words within each product:

In [ ]:
# Get the most common words for a product category
def get_top_words_by_product(product, n=10):

    # Select cleaned narratives for the chosen product
    product_text = text_df.loc[
        text_df["Product"] == product, "clean_narrative"
    ]

    # Combine the narratives into individual words
    words = " ".join(product_text).split()

    # Count the words
    word_counts = Counter(words)

    # Return the most frequent words
    return pd.DataFrame(
        word_counts.most_common(n),
        columns=["Word", "Frequency"]
    )


# Get the top 5 words for each product category
product_word_summary = []

# Loop through all 11 financial product categories
for product in text_df["Product"].unique():

    top_words_product = get_top_words_by_product(product, n=5)

    product_word_summary.append({
        "Product": product,
        "Top 5 Words": ", ".join(top_words_product["Word"])
    })

# Display the summary
product_word_summary = pd.DataFrame(product_word_summary)

product_word_summary

Raw counts are dominated by words that are common everywhere: "account" is in the top 5 for all 11 products and "credit" for 7. A few product words do break through (*mortgage*, *vehicle*, *mohela*, *collection*, *app*), and mortgage complaints name their product directly, which supports the masking test planned for Phase 3.

**TF-IDF scores** fix this: they reward words that are frequent in a complaint but rare across all complaints, so the top TF-IDF words for each product are the ones that mark it out. These are the same features our Phase 3 classifier will use. Longer bars mean more distinctive words (bar lengths compare words within a product, not across products).

In [ ]:
# Top 8 distinctive terms per product (highest average TF-IDF score)
distinct = distinctive_terms(text_df["clean"], text_df["product_short"], n=8)

# One small bar chart per product, arranged in a 3 x 4 grid
products = sorted(distinct["product"].unique())
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
for ax, product in zip(axes.flat, products):
    terms = distinct[distinct["product"] == product]   # this product's 8 terms
    sns.barplot(data=terms, x="score", y="term", color=MAIN, ax=ax)
    ax.set(title=product, xlabel="", ylabel="", xticks=[])   # scores only compare words within a product, so no x-axis
    sns.despine(ax=ax, left=True, bottom=True)
axes.flat[-1].axis("off")    # 11 products in a 12-slot grid: hide the empty one

fig.suptitle("Most products have their own vocabulary; credit reporting, debt collection and debt management share theirs",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

- **Eight products have clear, product-specific vocabulary:** mortgage (*escrow, property, servicing*), student loan (*school, mohela*), vehicle loan (*repossession, auto*), checking/savings (*deposit, check*), prepaid card (*gift card*). These should be easy to classify.
- **Credit reporting, debt collection and debt management share their top words** (*credit*, *report*). Debt management complaints are often about credit repair companies promising to fix a credit report, so their language overlaps with credit reporting. These categories share similar vocabulary, so we will examine whether the classification models confuse them more frequently in Phase 3.
- Company names show up as distinctive terms (*chime, mohela, affirm, paypal*). This confirms the point from 2.2: part of the signal is *who* the complaint is about, not just *what* it is about.

### 2.4 Word clouds: overlapping vs. distinct vocabulary

Word size shows how distinctive a word is for that product (TF-IDF). The three overlapping products sit beside mortgage, which has its own vocabulary.

In [ ]:
# Word clouds for the three overlapping products and one distinct product
cloud_products = ["Credit reporting", "Debt collection", "Debt management", "Mortgage"]
cloud_terms = distinctive_terms(text_df["clean"], text_df["product_short"], n=40)   # top 40 TF-IDF terms per product

def dark_blue(*args, **kwargs):
    # Colour every word in a readable dark blue (shades vary for texture)
    return f"hsl(215, 55%, {np.random.randint(25, 45)}%)"

# Draw one cloud per product in a 2 x 2 grid; word size = TF-IDF score
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, product in zip(axes.flat, cloud_products):
    weights = cloud_terms[cloud_terms["product"] == product].set_index("term")["score"].to_dict()   # {word: score}
    cloud = WordCloud(width=700, height=350, background_color="white", color_func=dark_blue,
                      prefer_horizontal=1.0, random_state=1).generate_from_frequencies(weights)
    ax.imshow(cloud, interpolation="bilinear")
    ax.set_title(product)
    ax.axis("off")   # hide the axes around the image

fig.suptitle('"Credit" and "report" dominate three different products; mortgage stands apart',
             x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

The first three clouds are dominated by the same words (*credit, report, reporting, debt*), while the mortgage cloud shares almost none of them. For the routing team this means a complaint that talks about a credit report could belong to any of three teams; the deciding clues are smaller words such as *validation* (debt collection) or *program, fees* (debt management).

### 2.5 Vocabulary Overlap Across Financial Products (Jaccard Similarity)

We use Jaccard similarity to measure how much vocabulary is shared between financial product categories.

Jaccard similarity compares the number of shared words with the total number of unique words across two sets.

A score of 0 indicates no shared words, while a score of 1 indicates identical word sets.

This analysis helps us identify product categories with similar vocabulary and explore which categories might be more difficult to distinguish during predictive modeling.

In [ ]:
# Get the 50 most common meaningful words for each financial product
top_words_by_product = {}

for product, group in text_df.groupby("Product"):

    # Combine the cleaned narratives and remove stopwords
    words = " ".join(group["clean"]).split()
    words = [word for word in words if word not in STOPWORDS]

    # Select the 50 most frequent words for each product
    word_counts = Counter(words)
    top_words_by_product[product] = set(
        word for word, count in word_counts.most_common(50)
    )

# Calculate Jaccard similarity between each pair of products
products = list(top_words_by_product.keys())

jaccard_matrix = pd.DataFrame(
    index=products,
    columns=products,
    dtype=float
)

for product_a in products:
    for product_b in products:
        words_a = top_words_by_product[product_a]
        words_b = top_words_by_product[product_b]

        # Count shared words and total unique words
        shared_words = len(words_a.intersection(words_b))
        total_unique_words = len(words_a.union(words_b))

        # Calculate Jaccard similarity
        jaccard_matrix.loc[product_a, product_b] = (
            shared_words / total_unique_words if total_unique_words else 0
        )

# Display the similarity scores
display(jaccard_matrix.round(2))

In [ ]:
# Shorten product names to make the heatmap easier to read
short_names = {
    "Checking or savings account": "Checking/Savings",
    "Credit card": "Credit Card",
    "Credit reporting or other personal consumer reports": "Credit Reporting",
    "Debt collection": "Debt Collection",
    "Debt or credit management": "Debt Management",
    "Money transfer, virtual currency, or money service": "Money Transfer",
    "Mortgage": "Mortgage",
    "Payday loan, title loan, personal loan, or advance loan": "Personal/Payday Loan",
    "Prepaid card": "Prepaid Card",
    "Student loan": "Student Loan",
    "Vehicle loan or lease": "Vehicle Loan"
}

# Apply shorter labels without changing the similarity scores
plot_matrix = jaccard_matrix.rename(
    index=short_names,
    columns=short_names
)

# Hide self-comparisons along the diagonal
mask = np.eye(len(plot_matrix), dtype=bool)

plt.figure(figsize=(13, 10))

sns.heatmap(
    plot_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor="white",
    square=True,
    cbar_kws={"label": "Jaccard Similarity"}
)

plt.title(
    "Vocabulary Similarity Across Financial Products\n"
    "Jaccard Similarity Based on Top 50 Words (Stopwords Removed)",
    fontsize=14,
    fontweight="bold",
    pad=20
)

plt.xlabel("Financial Product")
plt.ylabel("Financial Product")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

### Interpretation: Vocabulary Similarity Across Financial Products

The Jaccard similarity analysis compares the 50 most frequent words in each financial product category after removing stopwords.

- **Highest similarity (0.54):** Checking/Savings and Money Transfer, Money Transfer and Prepaid Card, and Personal/Payday Loan and Vehicle Loan share the most vocabulary. This makes sense because these products involve related financial activities.

- **Lowest similarity (0.09):** Credit Reporting and Prepaid Card have the least vocabulary overlap, suggesting that consumers use different language when describing problems with these products.

- **Credit Reporting and Debt Collection (0.47):** These categories share several common terms, which may make them harder for classification models to distinguish.

Overall, the results show that some financial products have distinct vocabularies, while others share similar language. This overlap may help explain potential classification errors when we evaluate our models in Phase 3.

### 2.6 Do complaints name their own product?

If most complaints simply say "my mortgage" or "my student loan", classification is easy but partly trivial. The chart shows, for complaints of each product (rows), the share that mention the key words of each product (columns). Numbers on the outlined diagonal mean complaints name their own product; strong numbers off the diagonal mean they mention another product's words.

In [ ]:
# % of complaints in each product (rows) that mention each product's key words (columns)
mentions = product_word_matrix(text_df)

# Heatmap: rows = actual product, columns = product whose key words are mentioned, cell = % of complaints
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(mentions, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "% of complaints"}, ax=ax)
ax.set(xlabel="Mentions the key words of...", ylabel="Complaint's actual product",
       title="Mortgage complaints name their product 83% of the time; debt management only 16%")

# Outline the diagonal: complaints mentioning their own product's words
for i in range(len(mentions)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor=ACCENT, linewidth=2))
plt.xticks(rotation=40, ha="right")   # tilt column names so they do not overlap
plt.tight_layout()
plt.show()

- **Some products almost always name themselves:** mortgage (83%), vehicle loan (73%), student loan (68%) and credit reporting (64%). For these, a model can score well just by spotting the product word, which is why Phase 3 will **re-test the model with these words masked** to see how much accuracy depends on them.
- **Others rarely do:** only 14% of payday/personal loan and 16% of debt management complaints use their own product's words. These classes depend on indirect clues and will likely be harder.
- **The key off-diagonal finding:** 33% of debt collection complaints and 28% of debt management complaints use credit-reporting words, as much as or more than their own. This matches 2.3 and 2.4: credit reporting, debt collection and debt management are the classes we expect the model to confuse, and the cases where the routing team would most benefit from a second look.

### 2.7 Sentiment Analysis Using VADER

We use VADER sentiment analysis to examine the emotional tone of consumer complaint narratives. VADER assigns sentiment scores based on the words and expressions in a text.

This analysis will help us understand the overall sentiment of complaints and compare sentiment across financial product categories.

In [ ]:
# Baseline VADER score for every complaint
text_df["sentiment_raw"] = sentiment_scores(text_df[TEXT_COL])            # compound score from -1 to +1
text_df["label_raw"] = sentiment_label(text_df["sentiment_raw"])          # Negative < -0.05 <= Neutral <= 0.05 < Positive

# Share of complaints in each sentiment label
print("Share of complaints by sentiment label (baseline VADER):")
print((text_df["label_raw"].value_counts(normalize=True) * 100).round(1).to_string())

- VADER calls about **36% of complaints positive**, even though every narrative here is a complaint. To find out why, we add up which positive dictionary words contribute most across all complaints.

In [ ]:
# Positive dictionary words that add the most to VADER scores
contributions = positive_word_contributions(text_df["clean"], n=12)   # top 12 words by total positive score added

# Horizontal bar chart of those words
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(data=contributions, x="positive_points", y="word", color=MAIN, ax=ax)
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=8)
ax.set(xlabel="Total positive points added across all complaints", ylabel="",
       title='"Credit" alone adds almost 10,000 positive points: VADER reads a topic word as praise')
sns.despine(left=True)
plt.tight_layout()
plt.show()

- VADER was built for social media, where *credit* means praise ("give credit") and *interest* means curiosity. In financial complaints these are **topic words**: *credit report*, *interest rate*, the **Fair** Credit Reporting Act, a loan **agreement**, even **Wells** Fargo. They inflate scores, most of all for credit-related products.

**Domain-adjusted sentiment analysis:** We make selected financial terms neutral in VADER's dictionary (`FINANCE_WORDS` in `src/text_analysis.py`) and calculate sentiment scores again. This is an experimental adjustment to reduce the influence of financial terminology, not a guaranteed improvement in sentiment accuracy.


In [ ]:
# Re-score with financial topic words made neutral
text_df["sentiment"] = sentiment_scores(text_df[TEXT_COL], neutral_words=FINANCE_WORDS)   # FINANCE_WORDS scored as 0
text_df["label"] = sentiment_label(text_df["sentiment"])

# Compare positive sentiment percentages before and after domain adjustment
positive_share = pd.DataFrame({
    "Baseline VADER": text_df["label_raw"].eq("Positive").groupby(text_df["product_short"]).mean() * 100,
    "Financial words neutral": text_df["label"].eq("Positive").groupby(text_df["product_short"]).mean() * 100,
}).sort_values("Baseline VADER", ascending=False)

# Reshape to long format (one row per product and version) so seaborn can draw paired bars
plot_data = positive_share.reset_index().melt(id_vars="product_short", var_name="version",
                                              value_name="pct_positive")
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=plot_data, x="pct_positive", y="product_short", hue="version",
            palette=["#c9c9c9", MAIN], ax=ax)
ax.set(xlabel="% of complaints labelled positive", ylabel="",
       title="Positive sentiment: Baseline VADER vs. domain-adjusted VADER")
ax.legend(title="", loc="lower right")
sns.despine(left=True)
plt.tight_layout()
plt.show()

# Compare the overall share of positive sentiment before and after domain adjustment
print("Overall share labelled positive:",
      f"baseline {text_df['label_raw'].eq('Positive').mean():.0%},",
      f"domain-adjusted {text_df['label'].eq('Positive').mean():.0%}")


- After making selected financial terms neutral, VADER classified approximately **77% of complaints as negative**, compared with the baseline results.
- The percentage classified as positive decreased, especially for credit reporting, debt collection, and credit card complaints.
- About **18% of complaints still received positive sentiment labels**. Words such as *help*, *resolve*, *support*, and *please* may contribute to these scores even when consumers are describing problems.
- **Takeaway:** Domain-specific terminology can influence sentiment scores. However, this adjustment is experimental, and we would need manually labeled sentiment data to determine whether it improves accuracy.


In [ ]:
# Share of complaints by sentiment label: baseline VADER vs. financial words neutral
order = ["Negative", "Neutral", "Positive"]   # fixed order for the x-axis
label_share = pd.DataFrame({
    "Baseline VADER": text_df["label_raw"].value_counts(normalize=True) * 100,
    "Financial words neutral": text_df["label"].value_counts(normalize=True) * 100,
}).reindex(order)

# Reshape to long format (one row per label and version) for paired bars
plot_data = label_share.reset_index(names="sentiment").melt(id_vars="sentiment", var_name="version",
                                                            value_name="pct")
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=plot_data, x="sentiment", y="pct", hue="version", order=order,
            palette=["#c9c9c9", MAIN], ax=ax)
for container in ax.containers:   # write the % above every bar
    ax.bar_label(container, fmt="%.0f%%", padding=3, fontsize=8)
ax.set(xlabel="", ylabel="% of complaints", ylim=(0, 100),
       title="Most complaints are negative, and more so once financial words are made neutral")
ax.legend(title="", loc="upper right")
sns.despine()
plt.tight_layout()
plt.show()

print(label_share.round(1).to_string())

- With baseline VADER, **60% of complaints are negative, 36% positive and 4% neutral**. Since every narrative is a complaint, a positive share this high is a warning sign.
- After financial topic words (*credit*, *interest*, *fair*, *agreement*) are made neutral, **77% of complaints are negative and 18% positive**. Almost all of the change moves from positive to negative; the neutral share stays small (4% to 5%).

In [ ]:
# Negative / neutral / positive mix for each product (financial words neutral)
# % of each product's complaints in each label (each row adds up to 100)
sentiment_by_product = (pd.crosstab(text_df["product_short"], text_df["label"], normalize="index") * 100)
sentiment_by_product = sentiment_by_product[["Negative", "Neutral", "Positive"]].sort_values("Negative")

# One stacked bar per product: negative + neutral + positive = 100%
ax = sentiment_by_product.plot(kind="barh", stacked=True, figsize=(10, 5.5), width=0.7,
                               color=["#4c72b0", "#c9c9c9", "#dd8452"])
# Label each segment with its %, skipping segments under 6% that are too narrow to read
for container in ax.containers:
    ax.bar_label(container, labels=[f"{v:.0f}%" if v >= 6 else "" for v in container.datavalues],
                 label_type="center", fontsize=8, color="white")
ax.set(xlim=(0, 100), xlabel="% of the product's complaints", ylabel="",
       title="Every product is mostly negative once financial words are made neutral")
ax.legend(title="", bbox_to_anchor=(1.01, 1), loc="upper left")
sns.despine(left=True)
plt.tight_layout()
plt.show()

print(sentiment_by_product.round(1).sort_values("Negative", ascending=False).to_string())

- **Every product is mostly negative** once financial words are made neutral. Debt collection has the highest negative share (86%) and student loan the lowest (67%, with the highest positive share at 28%).
- Debt collection and credit reporting have some of the highest negative sentiment percentages after the domain adjustment. Their positive sentiment percentages decreased substantially, suggesting that financial terminology influenced the original VADER scores. However, we cannot confirm that the original positive labels were incorrect without manually reviewing and labeling the complaints.

In [ ]:
# Does sentiment differ by product? (adjusted scores)
# Products ordered from most negative to least negative median score
order_by_sentiment = text_df.groupby("product_short")["sentiment"].median().sort_values().index

# Box = middle half of scores, line = median, whiskers = typical range (outliers hidden)
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=text_df, x="sentiment", y="product_short", order=order_by_sentiment,
            color=MAIN, showfliers=False, width=0.6, ax=ax)
ax.axvline(0, color="grey", linewidth=1, linestyle="--")   # neutral line
ax.set(xlabel="Sentiment score (−1 very negative, +1 very positive)", ylabel="",
       title="Every product is mostly negative, and the ranges overlap almost completely")
sns.despine(left=True)
plt.tight_layout()
plt.show()

# Does sentiment differ by how the company responded?
# Group company responses into three outcomes, then compare median sentiment and counts
outcome = text_df["Company response to consumer"].map(
    lambda r: "Relief given" if "relief" in r else ("Closed with explanation" if "explanation" in r else "Other"))
print(text_df.groupby(outcome)["sentiment"].agg(["median", "count"]).round(2))

- Median sentiment ranges only from about −0.8 (debt collection, money transfer) to −0.5 (student loan), and the boxes overlap heavily. **Sentiment will not help the model tell products apart**, so we will not use it as a classification feature in Phase 3.
- Sentiment is also the same whether the company gave relief or just closed with an explanation (both about −0.75). So for our stretch question, *how angry* a consumer sounds does not predict *whether they get relief*.
- Where sentiment could still help is in **prioritising** within a team's queue (the most negative complaints first). If we need a reliable severity or urgency score, Claude can judge it from context, which a word list cannot. We will test that in our Phase 3 plan.

### 2.8 Summary: what Phase 2 means for Phase 3

| Finding | Evidence | Phase 3 action |
|---|---|---|
| Balanced sample (454–455 per product); real complaints are 91% credit reporting | 1.3 | Report accuracy and macro F1; note that a balanced model over-predicts small products on real traffic |
| 105 exact copies of 44 template letters | 1.6 | Remove copies before the train/test split (4,895 narratives) |
| Most products have their own vocabulary | 2.3, 2.4 | TF-IDF features with logistic regression as the baseline |
| Credit reporting, debt collection and debt management share vocabulary | 2.3–2.5 | Per-class precision and recall, and a confusion matrix |
| Many complaints name their own product (mortgage 83%) | 2.6 | Re-train with product words masked and compare macro F1 |
| Company names are among the most common and distinctive terms | 2.2, 2.3 | Inspect top features; test a version with company names removed |
| Sentiment is mostly negative for every product and overlaps across products | 2.7 | Do not use sentiment as a classification feature |

## 3. Predictive modeling

**Goal.** Predict the **Product** (11 categories, Y) from the **Consumer complaint narrative** (X), so that complaints arriving without a product label (emails, chats, call notes) can be routed to the right team. The CFPB label is the product the consumer chose, so it stands in for the correct routing decision; it is not a verified routing outcome.

### How we measure success

**Accuracy** is our main measure: the share of complaints sent to the right team. Because errors cost each team differently, we also report each product team's outcomes:

| Outcome, for one product team (say, Mortgage) | What it means for routing |
|---|---|
| **True positive (TP)** | A mortgage complaint is sent to the mortgage team |
| **False positive (FP)** | The mortgage team receives a complaint about another product: extra work, and that consumer waits while it is re-routed |
| **False negative (FN)** | A mortgage complaint is sent to another team: the mortgage team never sees it, risking the response deadline |
| **True negative (TN)** | A complaint about another product is correctly kept away from the mortgage team |

**Precision** = TP / (TP + FP): how much of what a team receives is really theirs. **Recall** = TP / (TP + FN): how much of a team's work reaches it. **Specificity** = TN / (TN + FP): how well other products are kept away. **F1** combines precision and recall. **Macro** averages weight all 11 products equally.

### Seven approaches, plus a baseline

| # | Approach | Family | Why it is a candidate |
|---|---|---|---|
| 0 | Majority-class baseline | Baseline | The floor any useful model must beat |
| 1 | TF-IDF + multinomial Naïve Bayes | Trained on our labels | Fast, classic probabilistic text baseline |
| 2 | TF-IDF + logistic regression | Trained on our labels | One multinomial model for all 11 products, with one set of coefficients and a probability per product |
| 3 | TF-IDF + linear SVM | Trained on our labels | Usually the strongest linear model for sparse text |
| 4 | spaCy word vectors + Claude descriptions written from sample complaints | Embedding similarity | Our instructor's suggestion (class 6, Exercise 3): no trained classifier |
| 5 | MiniLM sentence embeddings + Claude descriptions written from the **product names only** | Embedding similarity | Sends no complaint text to Claude |
| 6 | MiniLM sentence embeddings + Claude descriptions written from sample complaints | Embedding similarity | Tests whether a better embedding fixes model 4 |
| 7 | Claude zero-shot (Haiku 5.5) | Direct LLM | Reads each complaint and names the product, with no training |

### How every value in this section is chosen

| Value | How it is chosen |
|---|---|
| 70/30 split, random seed 42 | Set in our proposal; leaves at least 119 complaints per product for testing |
| 5-fold stratified cross-validation, seed 42 | The same folds for every trained model, so differences come from the models |
| TF-IDF settings (models 1–3) | Rebecca's settings, chosen once and shared by all three models: words and two-word phrases (the proposal), English stop words removed, words in fewer than 3 training complaints or more than 95% of them dropped, log-scaled counts (`sublinear_tf`). They are **not tuned** (3.7) |
| Classifier settings | Naïve Bayes `alpha=1.0`, SVM `C=1.0` (scikit-learn defaults); `max_iter` only gives logistic regression (2,000) and the SVM (5,000) enough iterations to finish fitting |
| Sample complaints per product for models 4 and 6 (k) | 5, 10 or 20, chosen by accuracy on training complaints Claude did not see |
| Final model | The rule below, applied to training-set scores only |
| Words hidden in the leakage check | Every word in the CFPB product and sub-product names (3.11) |
| Groups in the subgroup check | Consumer tags in the data and quartiles of narrative length (3.13) |
| Hand-check size (30 complaints) | Set by the project brief |

**How the final model is chosen (decided before testing).** The approach with the highest accuracy on the training data (cross-validated for models 1–3, on training complaints the method never learned from for models 4–7). One exception: if logistic regression is within **one standard error** of the best (a difference inside the noise between folds), we choose logistic regression, because it gives probabilities that are easier to use for routing (3.12).

### How our instructor's feedback is addressed

| Feedback | Where it is handled |
|---|---|
| Phase 1: explain where the model is used when the category is unknown | The problem statement above: complaints arriving by email, chat or phone have no product label |
| Phase 1: mask explicit product words and see how much accuracy drops | 3.11, using every word in the CFPB product and sub-product names |
| Phase 1: results suited to 11 classes (macro F1, per-class precision and recall; one multinomial logistic regression) | 3.8 and 3.9: accuracy, macro averages, and TP / FP / FN / TN with precision, recall, specificity and F1 per product |
| Phase 1: add `.env` to `.gitignore` and an MIT license | Done in the repository |
| Phase 2: would embedding similarity with Claude-written category descriptions be a better, cheaper solution? | Models 4–6 (3.4, 3.5), compared with the others in 3.7 and 3.8 |
| Phase 2: generate `requirements.txt` with versions from the virtual environment | Done in the repository (`pip freeze`, kept to the packages the project imports) |

### 3.1 Train/test split

We split the 4,895 unique complaints 70% / 30%, as set in our proposal, **stratified** by product so every product has the same share in both sets. The rows are split (not separate text columns), so every approach is scored on exactly the same complaints, matched by Complaint ID. The test set is not used again until 3.8.

In [ ]:
SEED = 42   # fixed random seed, so every run gives the same split and results

# Stratified 70/30 split of the unique complaints (text_df from 2.1):
# every product keeps the same share in the training and test sets
train_df, test_df = train_test_split(text_df, test_size=0.30, stratify=text_df["Product"], random_state=SEED)

# The TF-IDF models read the original narratives, as in Rebecca's modelling (TfidfVectorizer lowercases them
# and removes English stop words itself); spaCy similarity (model 4) uses the cleaned text from 2.1
X_train, y_train = train_df[TEXT_COL], train_df["Product"]
X_test, y_test = test_df[TEXT_COL], test_df["Product"]

print(f"Training: {len(train_df):,} complaints   Test: {len(test_df):,} complaints")
pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}).rename(index=SHORT_NAMES)

Every product has 119 to 137 complaints in the test set, so each class is evaluated on enough examples. Credit reporting has the fewest (119) because most of the removed duplicate letters were credit reporting disputes.

### 3.2 Majority-class baseline

A model that ignores the text and always predicts the most common training product. Any useful model must beat it by a wide margin.

In [ ]:
# Always predict the most common product in the training data
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print(f"Always predicts: {baseline_pred[0]}")
print(f"Test accuracy: {accuracy_score(y_test, baseline_pred):.1%}   "
      f"Test macro F1: {f1_score(y_test, baseline_pred, average='macro'):.3f}")

Because the sample is balanced, the baseline is right only **9.3%** of the time (about 1 in 11), with a macro F1 of 0.016. On real complaint traffic, where 91% of complaints are credit reporting (1.3), the same baseline would score 91% accuracy while sending every other product to the wrong team. This is why we report macro F1 alongside accuracy.

### Evaluation tools used throughout section 3

In [ ]:
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)  # the same 5 folds for every comparison


def routing_table(y_true, y_pred):
    # One row per product: TP, FP, FN, TN and the rates built from them
    labels = sorted(pd.unique(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)       # rows = true product, columns = predicted
    tp = np.diag(cm)                                             # correct predictions for each product
    fp = cm.sum(axis=0) - tp                                     # predicted as this product but wrong
    fn = np.bincount(pd.Categorical(y_true, categories=labels).codes, minlength=len(labels)) - tp
    # fn: complaints of this product predicted as something else (incl. off-list Claude answers)
    tn = len(y_true) - tp - fp - fn                              # everything else
    table = pd.DataFrame({"TP": tp, "FP": fp, "FN": fn, "TN": tn}, index=labels)
    table["precision"] = (table["TP"] / (table["TP"] + table["FP"])).fillna(0)
    table["recall"] = table["TP"] / (table["TP"] + table["FN"])
    table["specificity"] = table["TN"] / (table["TN"] + table["FP"])
    table["F1"] = (2 * table["precision"] * table["recall"] / (table["precision"] + table["recall"])).fillna(0)
    return table


def summary(y_true, y_pred):
    # Accuracy plus the macro (equal-weight) averages of the per-product rates
    table = routing_table(y_true, y_pred)
    return {"accuracy": accuracy_score(y_true, y_pred),
            **{f"macro {m}": table[m].mean() for m in ["precision", "recall", "specificity", "F1"]}}


def cv_scores(pipeline):
    # 5-fold cross-validated accuracy (mean, spread and standard error) and macro F1, on the training set only;
    # the TF-IDF vocabulary is learned inside each fold, so the validation fold never shapes the features
    r = cross_validate(pipeline, X_train, y_train, cv=CV, scoring=["accuracy", "f1_macro"], n_jobs=-1)
    return {"CV accuracy": r["test_accuracy"].mean(),
            "accuracy SD": r["test_accuracy"].std(),
            "standard error": r["test_accuracy"].std() / np.sqrt(len(r["test_accuracy"])),
            "CV macro F1": r["test_f1_macro"].mean(),
            "macro F1 SD": r["test_f1_macro"].std()}


def holdout_accuracy(y_true, y_pred):
    # Accuracy and its standard error on a set of complaints the method never learned from
    acc = accuracy_score(y_true, y_pred)
    return {"CV accuracy": acc, "standard error": np.sqrt(acc * (1 - acc) / len(y_true))}

### Claude API tools used in sections 3 and 4

All the code that talks to Claude is in the cell below. It holds the models and prices, the **four prompts** (descriptions from sample complaints, descriptions from product names only, zero-shot classification, error analysis) and small functions that check the budget, send requests and **save every answer**, so the notebook re-runs without an API key and without paying again. The API key is read from `.env` (listed in `.gitignore`) and never printed.

In [ ]:
# ======================================================================
# CLAUDE API TOOLS
# This cell only DEFINES settings, prompts and functions; it sends nothing to Claude.
# Claude is only called in 3.4 (descriptions), 3.5 (label-only descriptions), 3.6 (zero-shot) and 4.3 (error analysis).
# Safety for our budget:
#   1. Every answer is saved to a CSV in Data/; if the CSV exists, Claude is never called again.
#   2. Before any batch is sent, its worst-case cost is estimated; if that would take total
#      spending over BUDGET_USD, the batch is NOT sent.
#   3. Every batch actually sent is recorded in Data/claude_spend_log.csv (date, task, tokens, cost).
# ======================================================================

# Settings
# Spending cap for this notebook. The team has $25 for the whole project; this notebook should cost
# about $1.50 in total, so $5 leaves room for one re-run without risking the rest of the budget
BUDGET_USD = 5.00

# File where every paid batch is recorded (committed to GitHub, so both team members see the total)
SPEND_LOG = DATA_DIR / "claude_spend_log.csv"

# The two Claude models we use: a small low-cost one and a larger, stronger one
CLAUDE_MODELS = {"Haiku 5.5": "claude-haiku-5-5", "Sonnet 5.5": "claude-sonnet-5-5"}

# Price in USD per million (input, output) tokens, from the model pages (November 2026):
# https://platform.claude.com/docs/en/models/haiku-5-5/overview  (rate for prompts under 100,000 tokens)
# https://www.anthropic.com/claude/sonnet
PRICES = {"claude-haiku-5-5": (0.10, 0.50), "claude-sonnet-5-5": (2.00, 10.00)}

# How many requests are sent at the same time (speeds up the run; lower it if the API reports rate limits)
PARALLEL_REQUESTS = 8

# The 11 official CFPB product names, taken from the data and listed in every prompt
PRODUCTS = sorted(text_df["Product"].unique())
CATEGORY_LIST = "\n".join(f"- {p}" for p in PRODUCTS)   # one product per line, as a bullet list

# ---------- Prompts ----------
# Prompt 1 (models 4 and 6): Claude reads k sample training complaints for ONE product
# and writes a short description of that product for a routing guide
DESCRIBE_PROMPT = """Below are {k} consumer complaints that were filed under the financial product category "{product}".

{samples}

Write a description of this product category for a complaint-routing guide: the kinds of accounts, problems and situations that complaints in this category describe. Use 3 to 5 plain sentences. Do not quote or summarise individual complaints, and do not mention the other categories."""

# Prompt 2 (model 5, Rebecca): Claude describes ONE product from its name only; no complaint text is sent
LABEL_PROMPT = """You are writing a reference definition for a financial consumer complaint classifier.
Category label: {product}
Describe the financial product/service, typical complaint issues, distinguishing terms, and
how it differs from related financial product categories. Use general domain knowledge only.
Do not invent case-specific details or refer to any individual consumer.
Write one concise paragraph of approximately 100–150 words. Return only the description."""

# Prompt 3 (model 7): Claude names the product of one complaint, with no training on our data
CLASSIFY_PROMPT = """You route consumer financial complaints to the right product team.

Product categories:
{categories}

Complaint:
<complaint>
{complaint}
</complaint>

Which ONE product category is this complaint about? Reply with the category name exactly as written above, and nothing else."""

# Prompt 4 (4.3): Claude reviews a complaint our model got wrong.
# Claude is NOT told the consumer's label or our model's prediction, so it cannot just agree with either
EXPLAIN_PROMPT = """Read the following consumer financial complaint and identify the most appropriate financial product category from the 11 categories provided.

Product categories:
{categories}

Complaint:
<complaint>
{complaint}
</complaint>

Answer on three lines:
CATEGORY: the category name exactly as written above
AMBIGUOUS: YES if the complaint could reasonably belong to more than one category, otherwise NO
CLUES: one or two sentences naming the words or context in the complaint that point to this category"""


# ---------- Cost: price, spending so far, and the budget check ----------
def cost_usd(input_tokens, output_tokens, model):
    # Dollar cost of a number of tokens for one model, at the prices above
    price_in, price_out = PRICES[model]
    return input_tokens / 1e6 * price_in + output_tokens / 1e6 * price_out


def spent_so_far():
    # Total dollars recorded in the spend log (0 if nothing has been spent yet)
    if not SPEND_LOG.exists():
        return 0.0
    return pd.read_csv(SPEND_LOG)["cost_usd"].sum()


def check_budget(prompts, model, max_tokens, task_name):
    # ~3 characters per token for these complaints (redactions like XXXX and numbers use more tokens),
    # plus a 50% safety margin, so the estimate stays above the real cost (checked on the first run: 4 was too low)
    est_in = sum(len(p) for p in prompts) / 3 * 1.5
    est_out = len(prompts) * max_tokens
    estimate = cost_usd(est_in, est_out, model)
    already = spent_so_far()
    print(f"{task_name}: {len(prompts):,} requests to {model}, at most ${estimate:.2f} "
          f"(spent so far ${already:.2f} of ${BUDGET_USD:.2f})")
    if already + estimate > BUDGET_USD:
        raise RuntimeError(f"Stopped before sending: ${already:.2f} + ${estimate:.2f} would exceed the "
                           f"${BUDGET_USD:.2f} budget. Nothing was sent and nothing was charged.")


def log_spending(task_name, model, results):
    # Add one line to the spend log with the REAL tokens and cost of a batch that was sent
    tokens_in = sum(r[1] for r in results)
    tokens_out = sum(r[2] for r in results)
    row = pd.DataFrame([{"date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "task": task_name,
                         "model": model, "requests": len(results), "input_tokens": tokens_in,
                         "output_tokens": tokens_out, "cost_usd": round(cost_usd(tokens_in, tokens_out, model), 4)}])
    row.to_csv(SPEND_LOG, mode="a", header=not SPEND_LOG.exists(), index=False)
    print(f"Done: actual cost ${row['cost_usd'].iloc[0]:.2f}; total spent ${spent_so_far():.2f}")


# ---------- Sending prompts ----------
def api_key():
    # The key from .env: Ketty's .env names it ANTHROPIC_API_KEY, Rebecca's ANTHROPIC_API_KEY_PROJECT
    load_dotenv(PROJECT_ROOT / ".env")
    return os.getenv("ANTHROPIC_API_KEY") or os.getenv("ANTHROPIC_API_KEY_PROJECT")


def get_client():
    # Connect to the API with the key from .env (the key is never printed).
    # Stops with a clear message if the key is missing.
    # max_retries=6: if the API is busy or rate-limits us, the client waits and tries again
    if not api_key():
        raise RuntimeError("No Claude API key found. Add ANTHROPIC_API_KEY to the .env file in the repo root.")
    return anthropic.Anthropic(api_key=api_key(), max_retries=6)


# How to switch off "thinking before answering" differs by model (Claude API docs, effort page):
# Haiku 5.5 accepts "disabled"; Sonnet 5.5 rejects it and uses "between_tools" instead
THINKING_OFF = {"claude-haiku-5-5": {"type": "disabled"}, "claude-sonnet-5-5": {"type": "between_tools"}}


def ask_claude(client, model, prompt, max_tokens):
    # Send ONE prompt and return (reply text, input tokens, output tokens).
    # Thinking is switched off: it counts toward max_tokens and is billed, and our tasks need a short direct answer
    # max_tokens caps the length of the reply, which also caps its cost.
    # Since API version 2026-10-21 `temperature` is no longer accepted, so answers can vary
    # slightly between runs: another reason every answer is saved
    response = client.messages.create(model=model, max_tokens=max_tokens,
                                      thinking=THINKING_OFF[model],
                                      messages=[{"role": "user", "content": prompt}])
    # A reply can contain several blocks (e.g. thinking, then text): keep only the text blocks
    text = "".join(block.text for block in response.content if block.type == "text")
    return text.strip(), response.usage.input_tokens, response.usage.output_tokens


def run_batch(prompts, model, max_tokens, task_name):
    # Send a list of prompts SAFELY: budget check first, then several requests at a time,
    # then record the real cost. Returns one (reply, input tokens, output tokens) per prompt, in order
    check_budget(prompts, model, max_tokens, task_name)      # stops here if over budget
    client = get_client()

    def one(prompt):
        try:
            return ask_claude(client, model, prompt, max_tokens)
        except Exception as error:                            # one failed request does not stop the batch
            return f"ERROR: {str(error)[:100]}", 0, 0

    with ThreadPoolExecutor(max_workers=PARALLEL_REQUESTS) as pool:
        results = list(pool.map(one, prompts))
    log_spending(task_name, model, results)
    return results


def check_access(models=CLAUDE_MODELS):
    # One tiny request per model to confirm the key works (about $0.00001 each, not logged)
    client = get_client()
    rows = []
    for name, model in models.items():
        try:
            reply, _, _ = ask_claude(client, model, "Reply with the word OK.", max_tokens=5)
            rows.append({"model": name, "model_id": model, "works": True, "message": reply})
        except Exception as error:                            # e.g. model not enabled for this key
            rows.append({"model": name, "model_id": model, "works": False, "message": str(error)[:200]})
    return pd.DataFrame(rows).set_index("model")


# ---------- Reading Claude's answers ----------
def match_category(answer):
    # Turn Claude's reply into one of the 11 official product names.
    # A reply that is not on the list is kept but flagged as UNMATCHED (it counts as wrong)
    answer = answer.strip().strip(".").strip()
    for product in PRODUCTS:
        if answer.lower() == product.lower():
            return product
    return f"UNMATCHED: {answer}"


def answer_fields(answer):
    # Split a reply like "CATEGORY: ...\nAMBIGUOUS: ...\nCLUES: ..." into a dictionary
    return {line.split(":", 1)[0].strip().upper(): line.split(":", 1)[1].strip()
            for line in answer.splitlines() if ":" in line}


# ---------- The three Claude tasks ----------
def describe_products(samples, model):
    # Models 4 and 6: one Claude-written description per product, from the sample complaints given for it
    groups = list(samples.groupby("Product"))                 # (product, its sample complaints)
    prompts = [DESCRIBE_PROMPT.format(
                   k=len(group), product=product,
                   samples="\n\n".join(f"Complaint {i + 1}:\n<complaint>\n{text}\n</complaint>"
                                       for i, text in enumerate(group[TEXT_COL])))
               for product, group in groups]
    results = run_batch(prompts, model, max_tokens=400, task_name=f"Product descriptions (k = {len(groups[0][1])})")
    return pd.DataFrame([{"Product": product, "k": len(group), "description": reply, "model": model,
                          "input_tokens": t_in, "output_tokens": t_out}
                         for (product, group), (reply, t_in, t_out) in zip(groups, results)])


def describe_from_labels(products, model):
    # Model 5 (Rebecca): one description per product from its name only (no complaint text is sent)
    results = run_batch([LABEL_PROMPT.format(product=p) for p in products], model, max_tokens=350,
                        task_name="Label-only product descriptions")
    return {p: reply for p, (reply, _, _) in zip(products, results) if not reply.startswith("ERROR")}


def classify_sample(sample, model):
    # Model 7: Claude names the product of every complaint in `sample` (zero-shot)
    prompts = [CLASSIFY_PROMPT.format(categories=CATEGORY_LIST, complaint=text) for text in sample[TEXT_COL]]
    results = run_batch(prompts, model, max_tokens=30, task_name="Zero-shot classification")
    return pd.DataFrame({"Complaint ID": sample["Complaint ID"].values,
                         "true_product": sample["Product"].values,
                         "claude_product": [match_category(r[0]) if not r[0].startswith("ERROR") else r[0]
                                            for r in results],
                         "model": model,
                         "input_tokens": [r[1] for r in results],
                         "output_tokens": [r[2] for r in results]})


def explain_errors(errors, model):
    # 4.3: for each complaint our model got wrong, Claude's product, an ambiguity flag and its clues.
    # `errors` needs the columns Complaint ID, Product (the consumer's label) and model_product
    prompts = [EXPLAIN_PROMPT.format(categories=CATEGORY_LIST, complaint=text) for text in errors[TEXT_COL]]
    results = run_batch(prompts, model, max_tokens=150, task_name="Error analysis")
    rows = []
    for (_, row), (reply, t_in, t_out) in zip(errors.iterrows(), results):
        fields = answer_fields(reply)                         # CATEGORY / AMBIGUOUS / CLUES
        rows.append({"Complaint ID": row["Complaint ID"], "consumer_product": row["Product"],
                     "model_product": row["model_product"],
                     "claude_product": match_category(fields.get("CATEGORY", reply)),
                     "claude_ambiguous": fields.get("AMBIGUOUS", "?").upper(),
                     "claude_clues": fields.get("CLUES", reply), "model": model,
                     "input_tokens": t_in, "output_tokens": t_out})
    return pd.DataFrame(rows)


def error_verdict(row):
    # Compare Claude's reading with the consumer's label and our model's prediction
    if row["claude_product"] == row["consumer_product"]:
        return "Model error"            # Claude agrees with the consumer: our model got it wrong
    if row["claude_product"] == row["model_product"]:
        return "Possible mislabel"      # Claude agrees with our model: the consumer's choice may be off
    return "Unclear"                    # Claude picks a third product


# ---------- Load saved answers, or call Claude once and save them ----------
def load_or_run(path, run):
    # If the answers are already saved in `path`, load them: no API call, no cost.
    # Otherwise call run() once (it goes through the budget check) and save the answers.
    # Without saved answers AND without an API key, skip the step so the rest of the notebook still runs
    if os.path.exists(path):
        return pd.read_csv(path)
    if not api_key():
        print(f"No saved results at {path} and no API key in .env: skipping this step.")
        return None
    results = run()
    results.to_csv(path, index=False)
    return results

### 3.3 Models 1–3: TF-IDF with Naïve Bayes, logistic regression and linear SVM

All three models use **the same TF-IDF settings** (Rebecca's, listed in the table at the start of section 3) and differ only in the classifier. Each is a `Pipeline`, so in cross-validation the vocabulary and IDF weights are learned from the four training folds only and never see the fold being scored.

In [ ]:
# Rebecca's TF-IDF settings, shared by all three models
TFIDF_SETTINGS = dict(stop_words="english",     # drop common English words ("the", "and")
                      ngram_range=(1, 2),       # single words and two-word phrases
                      min_df=3,                 # ignore terms in fewer than 3 training complaints
                      max_df=0.95,              # ignore terms in more than 95% of them
                      sublinear_tf=True)        # log-scaled counts, so repeated words count less

# Models 1-3: the same TF-IDF features, three classifiers
tfidf_models = {
    "Naïve Bayes": Pipeline([("tfidf", TfidfVectorizer(**TFIDF_SETTINGS)), ("clf", MultinomialNB(alpha=1.0))]),
    "Logistic regression": Pipeline([("tfidf", TfidfVectorizer(**TFIDF_SETTINGS)),
                                     ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    "Linear SVM": Pipeline([("tfidf", TfidfVectorizer(**TFIDF_SETTINGS)),
                            ("clf", LinearSVC(C=1.0, max_iter=5000, random_state=SEED))]),
}

# Size of the vocabulary these settings produce on the full training set
n_terms = len(TfidfVectorizer(**TFIDF_SETTINGS).fit(X_train).get_feature_names_out())
print(f"TF-IDF terms (words and two-word phrases) learned from the training complaints: {n_terms:,}")

# 5-fold cross-validation on the training set: accuracy (spread, standard error) and macro F1
tfidf_cv = pd.DataFrame({name: cv_scores(pipe) for name, pipe in tfidf_models.items()}).T
tfidf_cv.round(4)

- The settings produce **25,231 terms** (words and two-word phrases) from the training complaints.
- **Linear SVM is the most accurate in cross-validation: 72.6%** (macro F1 0.723), ahead of logistic regression (71.3%, 0.713) and Naïve Bayes (67.0%, 0.662).
- The SVM's lead over logistic regression (1.3 points) is larger than one standard error of the SVM (0.7 points), so it is more than fold-to-fold noise; Naïve Bayes is clearly behind.
- The spread across folds is small (accuracy SD 1.2–1.9 points), so the results do not depend on one lucky split.

### 3.4 Model 4: spaCy word vectors with Claude-written descriptions (instructor's suggestion)

This follows class 6, Exercise 3. Claude (Sonnet 5.5) reads **k sample training complaints** for each product and writes a short description of it (prompt 1). spaCy's `en_core_web_md` turns each description and each complaint into a vector (the average of its word vectors), and each complaint is assigned to the product whose description is most similar (cosine similarity). No classifier is trained on the labels.

**Choosing k:** 5, 10 or 20 complaints per product (doubling each time). The 20 sample complaints per product are drawn once from the training set; k = 5 and 10 use the first 5 and 10 of them. Each k is scored on the training complaints Claude did not see, and the most accurate k is kept. The test set is not used.

In [ ]:
# spaCy medium English model: the word vectors used in class 6 (only the vectors are needed, so other steps are skipped)
nlp = spacy.load("en_core_web_md", exclude=["tagger", "parser", "ner", "attribute_ruler", "lemmatizer", "senter"])


def embed(texts):
    # One 300-number vector per text: the average of its word vectors
    return np.vstack([doc.vector for doc in nlp.pipe(list(texts), batch_size=64)])


# Complaints are embedded from the cleaned text of 2.1 (about 30 seconds)
E_train, E_test = embed(train_df["clean"]), embed(test_df["clean"])
print(f"Embedded {len(E_train):,} training and {len(E_test):,} test complaints")

In [ ]:
# Candidate numbers of sample complaints Claude reads per product (doubling each time)
K_VALUES = [5, 10, 20]

# 20 random training complaints per product; k = 5 and k = 10 use the first 5 and 10 of them
description_samples = train_df.groupby("Product").sample(max(K_VALUES), random_state=SEED)
shown_to_claude = train_df["Complaint ID"].isin(description_samples["Complaint ID"])

# Claude (Sonnet 5.5) writes one description per product for each k (or the saved ones are loaded)
descriptions = load_or_run(
    DATA_DIR / "claude_product_descriptions.csv",
    lambda: pd.concat([describe_products(description_samples.groupby("Product").head(k), CLAUDE_MODELS["Sonnet 5.5"])
                       for k in K_VALUES], ignore_index=True))


def similarity_predict(E, description_table):
    # Assign each complaint to the product whose description vector is most similar
    D = embed(description_table["description"].map(clean_text))   # descriptions cleaned like the complaints
    similarity = normalize(E) @ normalize(D).T
    return np.array(description_table["Product"])[similarity.argmax(axis=1)], similarity


# Score each k on the training complaints Claude did not see
if descriptions is not None:
    held_out_E, held_out_y = E_train[~shown_to_claude.values], y_train[~shown_to_claude.values]
    rows = []
    for k in K_VALUES:
        pred, _ = similarity_predict(held_out_E, descriptions[descriptions["k"] == k])
        acc = accuracy_score(held_out_y, pred)
        rows.append({"k": k, "accuracy": acc, "standard error": np.sqrt(acc * (1 - acc) / len(held_out_y))})
    k_results = pd.DataFrame(rows).set_index("k")

    chosen_k = k_results["accuracy"].idxmax()     # the k with the highest accuracy
    print(f"Held-out training complaints: {len(held_out_y):,}   Chosen k = {chosen_k}")
    display(k_results.round(3))

In [ ]:
# Why can similarity struggle? How alike do complaints look to spaCy?
# Average cosine similarity of 2,000 random pairs of training complaints
rng = np.random.default_rng(SEED)
i, j = rng.integers(0, len(E_train), 2000), rng.integers(0, len(E_train), 2000)
pair_similarity = (normalize(E_train)[i] * normalize(E_train)[j]).sum(axis=1)
print(f"Average similarity between two random complaints: {pair_similarity.mean():.3f}  (1 = identical direction)")

if descriptions is not None:
    _, sim = similarity_predict(E_train, descriptions[descriptions["k"] == chosen_k])
    # Closest vs second-closest description: a small gap means the choice is nearly a coin flip
    top_two = np.sort(sim, axis=1)[:, -2:]
    print(f"Similarity to the closest description: {top_two[:, 1].mean():.3f}   "
          f"to the second closest: {top_two[:, 0].mean():.3f}")

Results (from the saved Claude results; shown when the notebook runs with the files in `Data/` and `outputs/`):
- With the best k (**k = 5**), spaCy similarity routes **18.0%** of the training complaints Claude never saw correctly: about twice the 9% baseline, far below the TF-IDF models. More examples did not help (k = 20 scored the same, k = 10 lower).
- **Why:** two random complaints have an average similarity of **0.948** in spaCy's averaged word vectors (printed above). Averaging hundreds of words makes every complaint point in almost the same direction, so a complaint is nearly as close to the second-best description as to the best one, and incidental words decide the product.

### 3.5 Models 5 and 6: MiniLM sentence embeddings

spaCy averages separate word vectors, which makes most complaints look alike (3.4). `all-MiniLM-L6-v2` (Sentence-Transformers) is trained so that texts with similar **meaning** get similar vectors. It runs locally, and we give it the original narratives. Two sets of Claude descriptions are compared:

- **Model 5, label-only descriptions (Rebecca):** Claude received only the 11 product names (prompt 2); **no complaint text was sent**. Rebecca generated these once and saved them; the saved file is loaded here.
- **Model 6, complaint-informed descriptions (Ketty):** the same descriptions as model 4, with k chosen again on training complaints Claude did not see.

The two description sets were made with different prompts and inputs, so the gap between models 5 and 6 does not isolate one cause.

In [ ]:
# Model 5 descriptions: Rebecca's label-only file (generated once with Claude, from the product names only)
LABEL_DESC_PATH = OUTPUT_DIR / "rebecca_modelling" / "claude_label_only_descriptions.json"
label_descriptions = json.loads(LABEL_DESC_PATH.read_text(encoding="utf-8")) if LABEL_DESC_PATH.exists() else {}
missing_labels = [p for p in PRODUCTS if p not in label_descriptions]
if missing_labels and api_key():
    # Only if the saved file is missing: generate the missing descriptions with Haiku 5.5 and save them
    label_descriptions.update(describe_from_labels(missing_labels, CLAUDE_MODELS["Haiku 5.5"]))
    LABEL_DESC_PATH.parent.mkdir(parents=True, exist_ok=True)
    LABEL_DESC_PATH.write_text(json.dumps(label_descriptions, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Label-only descriptions available: {sum(p in label_descriptions for p in PRODUCTS)} of {len(PRODUCTS)}")

# Load the sentence-embedding model (downloads once, about 90 MB); skip models 5 and 6 if it is not installed
try:
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer("all-MiniLM-L6-v2")
except Exception as error:
    sbert = None
    print("sentence-transformers is not available, so models 5 and 6 are skipped:", str(error)[:120])
    print("Install it once with:  %pip install sentence-transformers")

if sbert is not None:
    # One unit-length vector per complaint, from the original narratives
    S_train = sbert.encode(X_train.tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    S_test = sbert.encode(X_test.tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    print(f"Embedded {len(S_train):,} training and {len(S_test):,} test complaints")

In [ ]:
def sbert_predict(S, products, texts):
    # Assign each complaint to the product whose description vector is most similar (cosine similarity)
    D = sbert.encode(list(texts), normalize_embeddings=True)
    similarity = S @ D.T                              # vectors are unit length, so this is cosine similarity
    return np.array(products)[similarity.argmax(axis=1)]


# Model 5: label-only descriptions; Claude saw no complaints, so every training complaint can be scored
label_ready = sbert is not None and all(p in label_descriptions for p in PRODUCTS)
if label_ready:
    label_train_pred = sbert_predict(S_train, PRODUCTS, [label_descriptions[p] for p in PRODUCTS])
    label_score = holdout_accuracy(y_train, label_train_pred)
    print(f"Model 5 (label-only): training accuracy {label_score['CV accuracy']:.3f} "
          f"(standard error {label_score['standard error']:.3f}, {len(y_train):,} complaints)")

# Model 6: complaint-informed descriptions; choose k on the training complaints Claude did not see
if sbert is not None and descriptions is not None:
    held = ~shown_to_claude.values
    rows = []
    for k in K_VALUES:
        table = descriptions[descriptions["k"] == k]
        pred = sbert_predict(S_train[held], table["Product"], table["description"])
        rows.append({"k": k, **holdout_accuracy(y_train[held], pred)})
    sbert_k = pd.DataFrame(rows).set_index("k").rename(columns={"CV accuracy": "accuracy"})
    sbert_best_k = sbert_k["accuracy"].idxmax()
    print(f"Model 6 (complaint-informed): chosen k = {sbert_best_k}")
    display(sbert_k.round(3))

    # How separated are complaints now? (compare with spaCy in 3.4)
    rng = np.random.default_rng(SEED)
    i, j = rng.integers(0, len(S_train), 2000), rng.integers(0, len(S_train), 2000)
    print(f"Average similarity between two random complaints: {(S_train[i] * S_train[j]).sum(axis=1).mean():.3f}")

Results (from the saved Claude results; shown when the notebook runs with the files in `Data/` and `outputs/`):
- **Sentence embeddings fix most of spaCy's problem.** Two random complaints now have an average similarity of about **0.37** (spaCy: 0.948), so complaints are distinguishable.
- **Model 6 (complaint-informed descriptions)** keeps **k = 20** and scores about **49%** on training complaints Claude did not see, more than double spaCy. With better vectors, more examples now help.
- **Model 5 (label-only descriptions)** scores lower (its training accuracy is printed above; 43.5% on the test set in 3.8), but sends **no complaint text** to Claude.
- Both remain far below the trained TF-IDF models: a few sentences per product cannot hold what a model learns from 3,426 labelled complaints.

### 3.6 Model 7: Claude zero-shot

Claude Haiku 5.5 (the low-cost model) reads each complaint with the list of 11 products and names one (prompt 3 in 4.1). It is not trained on our data and has no settings to choose. We run it once on all 4,895 complaints and save the answers; its training-set score here is an honest estimate because it never learned from those complaints. The first run takes about 10 minutes.

In [ ]:
# Claude Haiku 5.5 names a product for every complaint (or the saved answers are loaded)
ZS_FILE = DATA_DIR / "claude_zero_shot_haiku.csv"
zero_shot = load_or_run(ZS_FILE, lambda: classify_sample(text_df, CLAUDE_MODELS["Haiku 5.5"]))

if zero_shot is not None:
    # Repair step (see the note below): replace only the answers that failed or were off-list.
    # It does nothing (and costs nothing) once every saved answer is a product name
    ids = zero_shot["Complaint ID"].astype(str)                       # compare IDs as text everywhere
    bad = zero_shot["claude_product"].str.startswith(("ERROR", "UNMATCHED"))
    if bad.any():
        # The failed requests of the first run were charged but logged at $0; add an estimate of their cost once:
        # the average input tokens of the requests that worked, and the full 30 output tokens each (worst case)
        failed = zero_shot["input_tokens"] == 0
        log = pd.read_csv(SPEND_LOG) if SPEND_LOG.exists() else pd.DataFrame(columns=["task"])
        if failed.any() and not log["task"].str.contains("estimated").any():
            avg_in = zero_shot.loc[~failed, "input_tokens"].mean()
            missing = cost_usd(avg_in * failed.sum(), 30 * failed.sum(), CLAUDE_MODELS["Haiku 5.5"])
            pd.DataFrame([{"date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
                           "task": "Zero-shot failed requests (estimated, not logged at the time)",
                           "model": CLAUDE_MODELS["Haiku 5.5"], "requests": int(failed.sum()),
                           "input_tokens": round(avg_in * failed.sum()), "output_tokens": int(30 * failed.sum()),
                           "cost_usd": round(missing, 4)}]).to_csv(SPEND_LOG, mode="a", header=False, index=False)
            print(f"Estimated unlogged cost of the first run added to the spend log: ${missing:.2f}")

        bad_ids = set(ids[bad])
        # Reuse the retried answers if they are still in memory from an earlier run of this cell (no new cost);
        # otherwise re-send only the bad complaints (goes through the budget check)
        if "retried" in globals() and bad_ids <= set(retried["Complaint ID"].astype(str)):
            print(f"Using the {len(retried):,} retried answers already in memory: nothing sent, no new cost")
        else:
            print(f"Re-sending {len(bad_ids):,} failed or off-list complaints")
            retried = classify_sample(text_df[text_df["Complaint ID"].astype(str).isin(bad_ids)],
                                      CLAUDE_MODELS["Haiku 5.5"])

        # Put each new answer in the row with the same Complaint ID (only the answer columns are replaced,
        # so the ID column keeps its original number type)
        new = retried.set_index(retried["Complaint ID"].astype(str))
        for col in ["claude_product", "model", "input_tokens", "output_tokens"]:
            zero_shot.loc[bad, col] = ids[bad].map(new[col]).values
        zero_shot.to_csv(ZS_FILE, index=False)                        # save the repaired answers
        print(f"Repaired file saved: {ZS_FILE.name}")

    # Look up Claude's answer for each training complaint and score it
    zero_shot_by_id = zero_shot.set_index(zero_shot["Complaint ID"].astype(str))["claude_product"]
    zs_train = zero_shot_by_id.reindex(train_df["Complaint ID"].astype(str)).values
    acc = accuracy_score(y_train, zs_train)
    # Answers that are not one of the 11 product names, or failed requests, count as wrong
    off_list = pd.Series(zs_train).str.startswith(("UNMATCHED", "ERROR")).sum()
    print(f"Training complaints: {len(zs_train):,}   accuracy: {acc:.3f} "
          f"(standard error {np.sqrt(acc * (1 - acc) / len(zs_train)):.3f})   off-list or failed answers: {off_list}")

**Note on the first zero-shot run (kept for the AI usage log).** The first run returned an error for **2,964 of 4,895** complaints (`'ThinkingBlock' object has no attribute 'text'`), and 4 more answers were off-list (Haiku corrected itself mid-answer: *"Wait, let me correct that..."*). The cause was in our code: Haiku 5.5 thinks before answering by default, so some replies started with a *thinking* block, and our code only read the first block. Thinking also counts toward `max_tokens` (30 here), so many replies were cut off before any answer. We switched thinking off, read only the text blocks, and re-sent **only** the affected complaints. Those failed requests were still charged but recorded with 0 tokens, so the cell above adds an estimate of their cost to the spend log.

Results (from the saved Claude results; shown when the notebook runs with the files in `Data/` and `outputs/`):
- **Claude zero-shot routes 71.0% of the training complaints correctly** with no training on our data, close to logistic regression (71.3%) and below the SVM (72.6%).
- **61 training answers (1.8%) were off-list** (not one of the 11 names); they count as wrong, so a system built on Claude needs a fallback for them.
- The score is an honest estimate because Claude never saw our labels; we cannot rule out that complaints like these were part of the public text Claude was trained on.

### 3.7 Comparing the seven approaches and choosing the final model

All scores below come from the **training set only**: cross-validation for models 1–3, and training complaints the method never learned from for models 4–7. The test set is still untouched.

In [ ]:
# Training-set accuracy of every approach before any choice is made (baseline scored the same way)
before = {"0. Majority baseline": cv_scores(Pipeline([("tfidf", TfidfVectorizer(**TFIDF_SETTINGS)),
                                                     ("clf", DummyClassifier())]))}
for i, name in enumerate(tfidf_models, start=1):
    before[f"{i}. TF-IDF + {name}"] = tfidf_cv.loc[name].to_dict()
if descriptions is not None:
    before["4. spaCy + complaint descriptions"] = {"CV accuracy": k_results.loc[chosen_k, "accuracy"],
                                                   "standard error": k_results.loc[chosen_k, "standard error"]}
if label_ready:
    before["5. MiniLM + label-only descriptions"] = label_score
if sbert is not None and descriptions is not None:
    before["6. MiniLM + complaint descriptions"] = {"CV accuracy": sbert_k.loc[sbert_best_k, "accuracy"],
                                                   "standard error": sbert_k.loc[sbert_best_k, "standard error"]}
if zero_shot is not None:
    before["7. Claude zero-shot"] = holdout_accuracy(y_train, zs_train)

before = pd.DataFrame(before).T[["CV accuracy", "standard error"]].rename(columns={"CV accuracy": "training-set accuracy"})

# Bar chart with ± 1 standard error, baseline in grey
fig, ax = plt.subplots(figsize=(10, 0.5 * len(before) + 1.5))
bars = ax.barh(before.index, before["training-set accuracy"] * 100, xerr=before["standard error"] * 100,
               color=[("#c9c9c9" if "baseline" in n else MAIN) for n in before.index], capsize=3)
ax.bar_label(bars, labels=[f"{a:.1%}" for a in before["training-set accuracy"]], padding=8, fontsize=9)
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel="Accuracy on training data (%), with ± 1 standard error",
       title="Training data: how often each approach routes a complaint correctly")
sns.despine(left=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
before.round(4)

In [ ]:
# The rule set at the start of section 3: highest training-set accuracy,
# but logistic regression if it is within one standard error of the best
candidates = before.drop(index="0. Majority baseline")
best = candidates["training-set accuracy"].idxmax()
lr = "2. TF-IDF + Logistic regression"
lr_within_noise = (candidates.loc[lr, "training-set accuracy"]
                   >= candidates.loc[best, "training-set accuracy"] - candidates.loc[best, "standard error"])
final_label = lr if lr_within_noise else best
print(f"Highest training-set accuracy: {best}")
print(f"Final model by our rule: {final_label}")

# The checks in 3.9 to 3.13 look inside the model (coefficients, scores, re-training on masked text),
# so they need one of our own TF-IDF models: the TF-IDF model picked by the same rule
tfidf_top = tfidf_cv["CV accuracy"].idxmax()
lr_close = (tfidf_cv.loc["Logistic regression", "CV accuracy"]
            >= tfidf_cv.loc[tfidf_top, "CV accuracy"] - tfidf_cv.loc[tfidf_top, "standard error"])
tfidf_best = "Logistic regression" if lr_close else tfidf_top
main_model = clone(tfidf_models[tfidf_best]).fit(X_train, y_train)   # fitted on the full training set
print(f"TF-IDF model analysed in 3.9 to 3.13: {tfidf_best}")

- **Linear SVM has the highest training-set accuracy (72.6%)**, followed by logistic regression (71.3%), Claude zero-shot (71.0%), Naïve Bayes (67.0%), MiniLM with complaint-informed descriptions (about 49%), MiniLM with label-only descriptions and spaCy (18.0%).
- Logistic regression is **not** within one standard error of the SVM (71.3% < 72.6% − 0.7% = 71.9%), so **by our rule the final model is TF-IDF + linear SVM**. It is also the model analysed in 3.9 to 3.13.
- The rule was applied before looking at the test set (3.8).

**Why the models are not tuned.** Every model keeps the settings listed at the start of section 3. Searching over settings such as the regularization strength C or the TF-IDF thresholds with cross-validation could raise accuracy, but it was not covered in this course and it is computationally expensive (each setting tried means re-training each model five times). We compared seven different approaches at fixed settings instead, which answers our main question (which kind of approach routes complaints best). Tuning the chosen model is a natural next step.

### 3.8 Test-set results: every approach, once

The 1,469 test complaints are now used for the first and only time. For models 1–3 we also show training accuracy, to check for overfitting.

In [ ]:
# Score every approach once on the test set
test_preds = {"0. Majority baseline": baseline.predict(X_test)}
fitted = {}
for i, (name, pipe) in enumerate(tfidf_models.items(), start=1):
    fitted[name] = clone(pipe).fit(X_train, y_train)                      # fit on all training data
    test_preds[f"{i}. TF-IDF + {name}"] = fitted[name].predict(X_test)
if descriptions is not None:
    test_preds["4. spaCy + complaint descriptions"] = similarity_predict(E_test, descriptions[descriptions["k"] == chosen_k])[0]
if label_ready:
    test_preds["5. MiniLM + label-only descriptions"] = sbert_predict(S_test, PRODUCTS, [label_descriptions[p] for p in PRODUCTS])
if sbert is not None and descriptions is not None:
    table = descriptions[descriptions["k"] == sbert_best_k]
    test_preds["6. MiniLM + complaint descriptions"] = sbert_predict(S_test, table["Product"], table["description"])
if zero_shot is not None:
    # Claude's saved answers matched by Complaint ID; off-list answers count as wrong
    test_preds["7. Claude zero-shot"] = zero_shot_by_id.reindex(test_df["Complaint ID"].astype(str)).fillna("MISSING").values

# Accuracy and macro precision / recall / specificity / F1 for each approach
test_results = pd.DataFrame({name: summary(y_test, pred) for name, pred in test_preds.items()}).T

# Training accuracy for the three trained models, to check for overfitting
test_results["training accuracy"] = pd.Series({f"{i}. TF-IDF + {n}": accuracy_score(y_train, fitted[n].predict(X_train))
                                               for i, n in enumerate(tfidf_models, start=1)})
test_results.to_csv(OUTPUT_DIR / "test_metrics.csv")
test_results.round(3)

In [ ]:
# Test accuracy and macro F1 for every approach
# Long format for seaborn: one row per approach and measure
plot_data = (test_results[["accuracy", "macro F1"]].reset_index(names="model")
             .melt(id_vars="model", var_name="measure", value_name="score"))
fig, ax = plt.subplots(figsize=(10, 0.6 * len(test_results) + 1.5))
sns.barplot(data=plot_data, x="score", y="model", hue="measure", palette=[MAIN, "#9ecae1"], ax=ax)
for container in ax.containers:                      # write the score at the end of each bar
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)
ax.set(xlim=(0, 1.05), xlabel="Score on the 1,469 test complaints", ylabel="",
       title="Test-set accuracy and macro F1, every approach")
ax.legend(title="", loc="lower right")
sns.despine(left=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

- **Linear SVM (final model): 73.0% test accuracy, macro F1 0.728.** Logistic regression: 73.1% / 0.731. Claude zero-shot: 72.6% / 0.724. Naïve Bayes: 70.2% / 0.696. Baseline: 9.3% / 0.016. The embedding approaches are far behind: MiniLM with complaint-informed descriptions 48.7%, label-only 43.5%, spaCy 18.3% (Claude-based rows appear when the saved results are present).
- **The top three are tied on the test set:** the 0.1–0.4 point differences are well inside the uncertainty of a 1,469-complaint test (about ±2.3 points at 95%). We keep the SVM because it was chosen on training data before testing; picking logistic regression now, for a 0.1-point test lead, would be choosing on the test set.
- **Test scores match cross-validation** (SVM 72.6% → 73.0%, logistic regression 71.3% → 73.1%), so cross-validation gave an honest estimate.
- **Overfitting check.** The SVM scores **99.9%** on the complaints it was trained on and 73.0% on new ones; logistic regression 92.6% vs 73.1%; Naïve Bayes 88.7% vs 70.2%. The SVM memorises its training complaints almost completely, which is common with 25,000 text features and 3,426 complaints, but its test score matches its cross-validated score, so it is not overfitting in a way that hurts new complaints. It is not underfitting either: it is 64 points above the baseline.
- **Macro specificity 0.973:** on average, each product team is kept clear of 97% of the complaints that are not theirs.
- **Claude zero-shot has the highest macro precision (0.775)**: when it sends a complaint to a team, it is most often really theirs.

### 3.9 Product by product

TP, FP, FN and TN for each product team on the test set, with the rates built from them (definitions at the start of section 3): first for the TF-IDF model chosen in 3.7, then for Claude zero-shot.

In [ ]:
# Test-set predictions of the chosen TF-IDF model, and each product team's TP / FP / FN / TN with the rates
test_pred = pd.Series(main_model.predict(X_test), index=y_test.index)
final_table = routing_table(y_test, test_pred).rename(index=SHORT_NAMES).sort_values("F1")
final_table.to_csv(OUTPUT_DIR / "per_product_tfidf.csv")

# Show the rates with 2 decimals and mark the weakest product in each rate column
# (highlight_min does not need matplotlib, unlike background_gradient)
final_table.style.format({c: "{:.2f}" for c in ["precision", "recall", "specificity", "F1"]}) \
    .highlight_min(subset=["precision", "recall", "specificity", "F1"], color="#f4cccc")

In [ ]:
# Confusion matrix: rows = true product, columns = predicted product
# Colour = share of the true product's complaints; numbers = complaint counts
# Products grouped by family (credit & debt, payments, loans) so related products sit side by side
labels = ["Credit reporting", "Debt collection", "Debt management",
          "Checking/savings", "Money transfer", "Prepaid card", "Credit card",
          "Mortgage", "Vehicle loan", "Payday/personal loan", "Student loan"]
counts = pd.crosstab(y_test.map(SHORT_NAMES), test_pred.map(SHORT_NAMES)).reindex(index=labels, columns=labels, fill_value=0)
shares = counts.div(counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(shares, annot=counts, fmt="d", cmap="Blues", vmin=0, vmax=100, linewidths=0.5,
            linecolor="white", cbar_kws={"label": "% of the true product's complaints"}, ax=ax)
for i in range(len(labels)):                             # outline the correct predictions
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor=ACCENT, linewidth=2))
for start, size in [(0, 3), (3, 4), (7, 4)]:            # outline each product family
    ax.add_patch(plt.Rectangle((start, start), size, size, fill=False, edgecolor="black", linewidth=1.2, linestyle="--"))
ax.set(xlabel="Predicted product", ylabel="True product",
       title="Most errors stay within a product family (dashed boxes): credit & debt, payments, loans")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

# How many errors stay inside a product family?
family = {p: f for f, group in [("credit & debt", labels[:3]), ("payments", labels[3:7]), ("loans", labels[7:])] for p in group}
is_error = test_pred != y_test
same_family = test_pred.map(SHORT_NAMES).map(family) == y_test.map(SHORT_NAMES).map(family)
print(f"Test errors: {is_error.sum()}   within the same family: {(is_error & same_family).sum()} "
      f"({(is_error & same_family).sum() / is_error.sum():.0%})")

In [ ]:
# Claude zero-shot: each product team's TP / FP / FN / TN on the test set (saved answers, no API cost)
if zero_shot is not None:
    claude_test = pd.Series(test_preds["7. Claude zero-shot"], index=y_test.index)
    claude_table = routing_table(y_test, claude_test).rename(index=SHORT_NAMES).sort_values("F1")
    # Off-list answers belong to no product: they appear as false negatives only
    print(f"Off-list or missing answers on the test set: {claude_test.str.startswith(('UNMATCHED', 'ERROR', 'MISSING')).sum()}")
    display(claude_table.style.format({c: "{:.2f}" for c in ["precision", "recall", "specificity", "F1"]})
            .highlight_min(subset=["precision", "recall", "specificity", "F1"], color="#f4cccc"))

- **Strongest teams: Mortgage** (F1 0.88; precision and recall 0.88) and **Student loan** (F1 0.88; precision 0.89). Both have vocabulary no other product uses (escrow, foreclosure, forbearance; see 3.10).
- **Credit reporting** has high recall (0.87: only 16 of its 119 complaints go elsewhere) but precision of 0.69: it receives **46 complaints that are not its own**, from products whose complaints also mention credit reports.
- **Weakest team: Debt management** (F1 0.59). It misses **65 of its 135 complaints** (recall 0.52), most often to debt collection (13).
- **Specificity is 0.96 or higher for every product.**
- **55% of the 397 errors stay inside a product family** (dashed boxes). The largest single confusions are among payment products: prepaid card sent to money transfer (19), money transfer to checking/savings (16), checking/savings to money transfer (14) and to prepaid card (13). These products share apps, cards and accounts (Cash App, PayPal, debit cards).
- **For routing:** mortgage, student loan and vehicle loan predictions can be trusted most; complaints routed to debt management, credit card and the three payment products are the ones most worth a second look.
- The second table shows the same breakdown for Claude zero-shot, for comparison.

### 3.10 Which words drive the predictions

The chosen TF-IDF model learns **one coefficient per term (word or two-word phrase) for each product**; a large positive coefficient pushes a complaint towards that product. Below are the 10 strongest words per product. Words that are **company names** are shown in red, because a model that relies on them may fail on a company it has not seen.

To flag company words without making up a list, we use the `Company` column of our data: a top word counts as a **company word** if it appears in a company's name **and most complaints that use it (more than half) were filed against that company**. That catches *mohela* or *synchrony*, but not ordinary words that happen to be in one company's name (*card*, *savings*). Consumers often use brand names that differ from legal names (*Cash App* belongs to *Block, Inc.*), so this flag finds fewer company words than there really are.

In [ ]:
# A word is a company word if it is in a company's name AND most complaints using it are against that company
company_lower = text_df["Company"].fillna("").str.lower()


def is_company_word(word):
    in_text = text_df["clean"].str.contains(rf"\b{re.escape(word)}\b")
    if word in ENGLISH_STOP_WORDS or not company_lower.str.contains(rf"\b{re.escape(word)}\b").any() or in_text.sum() == 0:
        return False
    return company_lower[in_text].str.contains(rf"\b{re.escape(word)}\b").mean() > 0.5


# Top 10 words per product: the largest coefficients in each product's row of the model
vocab = np.array(main_model.named_steps["tfidf"].get_feature_names_out())
clf = main_model.named_steps["clf"]
feats = pd.DataFrame([{"product": SHORT_NAMES[p], "term": vocab[j], "coefficient": clf.coef_[i][j]}
                      for i, p in enumerate(clf.classes_) for j in np.argsort(clf.coef_[i])[::-1][:10]])
feats["is_company"] = feats["term"].map(lambda t: all(is_company_word(w) for w in t.split()))

# One small bar chart per product; company words in red
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
for ax, product in zip(axes.flat, sorted(feats["product"].unique())):
    rows = feats[feats["product"] == product]
    ax.barh(rows["term"], rows["coefficient"], color=[ACCENT if c else MAIN for c in rows["is_company"]])
    ax.invert_yaxis()
    ax.set(title=product, xlabel="", ylabel="", xticks=[])
    sns.despine(ax=ax, left=True, bottom=True)
axes.flat[-1].axis("off")
fig.suptitle("Top words per product; company names in red", x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Company words among the top 10 words: {feats['is_company'].sum()} of {len(feats)}: "
      f"{', '.join(sorted(feats.loc[feats['is_company'], 'term']))}")

- **Domain sense.** Most top terms are the product's own vocabulary: *escrow, foreclosure, modification* (mortgage), *repossession, dealership* (vehicle loan), *forbearance, school, education* (student loan), *overdraft* (checking/savings), *validation, owed* (debt collection), *inquiries, inaccuracies, late payments* (credit reporting), *debt relief, credit repair* (debt management), *gift card* (prepaid card), *recipient, Zelle, seller* (money transfer). *Solar* and *installation* under payday/personal loan make sense: solar-panel financing is filed as a personal loan.
- **Red flag 1: company names.** **21 of the 110 top terms are company names** (shown in red), and some rank first: *mohela* for student loan, *paypal* for money transfer, *affirm* and *klarna* for payday/personal loan. The model partly learns **who** the complaint is about rather than **what** it is about, so complaints about a new or smaller company get less help. The flag undercounts: *citi* and *cashapp* are not flagged because the companies' legal names differ.
- **Red flag 2: formatting artifacts.** *xxxx balance* (credit reporting) and *00 loan* (payday/personal loan) come from CFPB redactions and dollar amounts (`{$500.00}`) in the original text. They are real patterns in this data but would not exist in a company's own emails, so they should be removed before deployment.
- **What it means:** these terms are **diagnostic, not actionable**: nobody changes how consumers write. They are used to trigger routing, and accuracy should be re-checked whenever new companies start appearing in complaints.

### 3.11 Leakage check: hiding the product names

If the model only works because consumers write "my mortgage" or "my credit card", it would fail on complaints that do not name the product. We re-train the chosen TF-IDF model (same settings) with **every word that appears in the CFPB product and sub-product names** replaced by a neutral token in both training and test text. The list comes straight from the data's own category names, minus common English words.

In [ ]:
# Every word in the product and sub-product names (e.g. mortgage, credit, card, student, loan, prepaid)
category_names = pd.concat([text_df["Product"], text_df["Sub-product"]]).dropna().unique()
product_name_words = sorted(set(re.findall(r"[a-z]+", " ".join(category_names).lower())) - ENGLISH_STOP_WORDS)
print(f"{len(product_name_words)} words masked:", ", ".join(product_name_words))


def mask_words(text, words, token="maskedterm"):
    # Replace each listed word (and its plural) with a neutral token
    return re.sub(r"\b(?:" + "|".join(words) + r")s?\b", token, text, flags=re.IGNORECASE)


# Re-train the same model on masked training text and predict on masked test text
masked_model = clone(main_model).fit(X_train.map(lambda s: mask_words(s, product_name_words)), y_train)
masked_pred = masked_model.predict(X_test.map(lambda s: mask_words(s, product_name_words)))

# Compare overall scores with and without the product-name words
masking = pd.DataFrame({"Original": summary(y_test, test_pred), "Product names masked": summary(y_test, masked_pred)}).T
has_name_word = X_test.str.contains(r"\b(?:" + "|".join(product_name_words) + r")s?\b", case=False).mean()
print(f"Test complaints containing at least one masked word: {has_name_word:.0%}")
masking.round(3)

In [ ]:
# Recall per product (share of each team's complaints that reach it): original vs. masked
compare = pd.DataFrame({"Original": routing_table(y_test, test_pred)["recall"],
                        "Product names masked": routing_table(y_test, masked_pred)["recall"]}).rename(index=SHORT_NAMES)
compare = compare.sort_values("Original")

fig, ax = plt.subplots(figsize=(10, 5.5))
# Dumbbell chart: grey dot = masked, blue dot = original
ax.hlines(compare.index, compare.min(axis=1), compare.max(axis=1), color="#e0e0e0", linewidth=3)
ax.scatter(compare["Product names masked"], compare.index, color="#9a9a9a", s=55, zorder=3, label="Product names masked")
ax.scatter(compare["Original"], compare.index, color=MAIN, s=60, zorder=3, label="Original")
ax.set(xlim=(0, 1), xlabel="Recall on the test set", ylabel="", title="Recall by product with and without the product names")
ax.legend(loc="lower right")
sns.despine(left=True)
plt.tight_layout()
plt.show()
compare.assign(change=compare["Product names masked"] - compare["Original"]).round(2)

- **90 words** are masked, and **98% of test complaints contain at least one**, so the check removes most explicit product clues.
- Accuracy falls from **73.0% to 67.2%** (−5.8 points) and macro F1 from 0.728 to 0.672. Most of the model's skill comes from the **context** of the complaint (what happened, which fees, which documents), not from the product's name.
- The biggest drops are **credit card** (recall 0.67 → 0.54) and **mortgage** (0.88 → 0.79), whose complaints most often name the product; payday/personal loan, checking/savings, prepaid card and money transfer each drop about 0.07–0.08. Debt management barely changes (0.52 → 0.51): it rarely names itself (2.6).
- **For routing:** complaints that never name their product (a short phone note, a chat message) can still be routed at about two in three.

### 3.12 Using the model's confidence for routing

A team could route the predictions the model is sure about automatically and send the rest to a person. For logistic regression, confidence is its highest probability; a linear SVM gives scores, not probabilities, so its confidence is the **gap between its top two scores** (a larger gap = a clearer choice). **We do not pick a cut-off**: that depends on how much manual review the team can handle. We show the trade-off at every 10% step, with cut-offs set on cross-validated training predictions; the test set then checks that the same cut-offs behave the same way on new complaints.

In [ ]:
# Confidence score: highest probability (logistic regression) or gap between the top two scores (SVM)
method = "predict_proba" if hasattr(main_model, "predict_proba") else "decision_function"


def confidence(scores):
    # Highest probability, or the gap between the highest and second-highest score
    if method == "predict_proba":
        return scores.max(axis=1)
    top_two = np.sort(scores, axis=1)[:, -2:]
    return top_two[:, 1] - top_two[:, 0]


# Cross-validated scores on the training set (each complaint scored by a model that did not see it)
classes = np.sort(y_train.unique())
cv_out = cross_val_predict(clone(main_model), X_train, y_train, cv=CV, method=method, n_jobs=-1)
cv_conf = confidence(cv_out)
cv_correct = classes[cv_out.argmax(axis=1)] == y_train.values

# The same confidence on the test set, from the model trained on all training data
test_conf = confidence(getattr(main_model, method)(X_test))
test_correct = (test_pred == y_test).values

rows = []
for share in np.arange(1.0, 0.05, -0.1):                 # auto-route 100%, 90%, ..., 10% of complaints
    cutoff = np.quantile(cv_conf, 1 - share)              # confidence needed to be in the top share
    rows.append({"share auto-routed": round(share, 1), "confidence cut-off": cutoff,
                 "accuracy of auto-routed (CV, training)": cv_correct[cv_conf >= cutoff].mean(),
                 "share auto-routed (test)": (test_conf >= cutoff).mean(),
                 "accuracy of auto-routed (test)": test_correct[test_conf >= cutoff].mean()})
tradeoff = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(tradeoff["share auto-routed"] * 100, tradeoff["accuracy of auto-routed (CV, training)"] * 100,
        marker="o", color=MAIN, linewidth=2, label="Cross-validated (training)")
ax.plot(tradeoff["share auto-routed (test)"] * 100, tradeoff["accuracy of auto-routed (test)"] * 100,
        marker="s", color=ACCENT, linewidth=1.5, linestyle="--", label="Test set")
ax.invert_xaxis()
ax.set(xlabel="Share of complaints routed automatically (%)", ylabel="Accuracy of those routed (%)",
       ylim=(60, 100), title="Routing fewer complaints automatically makes those routed more accurate")
ax.legend(loc="lower right")
sns.despine()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "routing_tradeoff.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Confidence used: {'highest probability' if method == 'predict_proba' else 'gap between the top two SVM scores'}")
tradeoff.round(3)

- The SVM's confidence is the gap between its top two scores (not a probability).
- Routing **every** complaint automatically gives 72.6% accuracy in cross-validation (73.0% on the test set).
- Routing only the **most confident half** raises accuracy to **90.9%** in cross-validation and **90.7%** on the test set; the **most confident 20%** reach 95.8% and 98.0%.
- The test curve follows the cross-validated one within about two points, so cut-offs chosen on training data hold on new complaints (52% of test complaints pass the cut-off set for 50%).
- **For the routing team:** this table is the menu. If reviewers can handle half the volume, the other half is routed automatically at about 91% accuracy; a team with less capacity moves down the table. Before production, the scores should be calibrated and the cut-off confirmed on fresh validation data.

### 3.13 Does the model work equally well for everyone?

We compare test accuracy across the **consumer tags** in the data (older Americans, servicemembers) and across **quartiles of narrative length** (four equal-sized groups, from shortest to longest), so the groups come from the data rather than from cut-offs we pick.

In [ ]:
# 1 if the final model routed the test complaint correctly, 0 if not
correct = (test_pred == y_test).values


def accuracy_by_group(groups):
    # Accuracy, size and 95% margin of error for each group
    out = pd.DataFrame({"group": groups, "correct": correct}).groupby("group", observed=True)["correct"]
    table = pd.DataFrame({"accuracy": out.mean(), "complaints": out.size()})
    table["± margin (95%)"] = 1.96 * np.sqrt(table["accuracy"] * (1 - table["accuracy"]) / table["complaints"])
    return table


# Groups 1: the consumer tags in the data
by_tag = accuracy_by_group(test_df["Tags"].fillna("No tag").values).sort_values("complaints", ascending=False)
# Groups 2: four equal-sized groups by narrative length, labelled with their word ranges
length_quartile = pd.qcut(test_df["word_count"], 4)
length_quartile = length_quartile.cat.rename_categories(
    [f"{int(max(i.left, 0)) + 1}-{int(i.right)} words" for i in length_quartile.cat.categories])
by_length = accuracy_by_group(length_quartile.values)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, table, title in [(axes[0], by_tag, "By consumer tag"), (axes[1], by_length, "By narrative length (quartiles)")]:
    ax.barh(table.index.astype(str), table["accuracy"] * 100, xerr=table["± margin (95%)"] * 100, color=MAIN, capsize=3)
    for y, (a, n) in enumerate(zip(table["accuracy"], table["complaints"])):
        ax.text(2, y, f"{a:.0%} (n={n:,})", va="center", color="white", fontsize=9, fontweight="bold")
    ax.invert_yaxis()
    ax.set(xlim=(0, 105), xlabel="Test accuracy (%), with 95% margin of error", ylabel="", title=title)
    sns.despine(ax=ax, left=True)
fig.suptitle("Accuracy by consumer group", x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
display(by_tag.round(3), by_length.round(3))

- **Narrative length matters most.** The shortest quarter of complaints (8–91 words) is routed correctly **62.8%** of the time, the longest quarter (287+ words) **79.2%**. The 95% margins do not overlap (at most 67.7% vs at least 75.0%), so this 16-point gap is real: short complaints carry fewer clues.
- **Servicemember complaints: 68.6%** (153 complaints, ± 7.4) vs **73.8%** for untagged ones (1,225, ± 2.5). The margins overlap, so we cannot say the gap is real; it is worth monitoring. Older American complaints (69.6%, 69 complaints, ± 10.9) are too few to judge.
- **For routing:** short complaints, such as chat messages and phone notes, are the ones to send to a person, or to follow up with a question about the product.
- These are descriptive comparisons on the groups the data records; they are not a full fairness audit.

## 4. Tool integration: Claude

Claude is used in four places:

| Task | Model | Where | Complaint text sent? |
|---|---|---|---|
| Describe each product from sample training complaints | Sonnet 5.5 | Models 4 and 6 (3.4, 3.5) | Yes, 5–20 per product |
| Describe each product from its name only | Haiku (Rebecca's run) | Model 5 (3.5) | No |
| Classify every complaint with no training | Haiku 5.5 | Model 7 (3.6) | Yes, all 4,895 |
| Explain the chosen model's mistakes | Haiku 5.5 | 4.3 | Yes, the test errors |

All the API code is in the cell *Claude API tools used in sections 3 and 4* (before 3.3). Every answer is saved the first time it runs, so the notebook re-runs without an API key and without paying again.

### 4.1 Prompts

In [ ]:
# Print the four prompts sent to Claude
for title, prompt in [("PROMPT 1: DESCRIPTIONS FROM SAMPLE COMPLAINTS (models 4 and 6)", DESCRIBE_PROMPT),
                      ("PROMPT 2: DESCRIPTIONS FROM PRODUCT NAMES ONLY (model 5)", LABEL_PROMPT),
                      ("PROMPT 3: ZERO-SHOT CLASSIFICATION (model 7)", CLASSIFY_PROMPT),
                      ("PROMPT 4: ERROR ANALYSIS (4.3)", EXPLAIN_PROMPT)]:
    print(title + "\n" + "-" * 60 + "\n" + prompt + "\n\n")

The error-analysis prompt deliberately does **not** tell Claude which product the consumer chose or what our model predicted, so it cannot simply agree with either one. The label-only prompt contains no complaint text at all.

### 4.2 Reviewing Claude's product descriptions

Models 4–6 depend entirely on these descriptions, so we read them: the complaint-informed set for the chosen k, and Rebecca's label-only set.

In [ ]:
# Complaint-informed descriptions (models 4 and 6), for the k chosen in 3.4
pd.set_option("display.max_colwidth", None)
if descriptions is not None:
    display(descriptions[descriptions["k"] == chosen_k][["Product", "description"]].set_index("Product")
            .rename(index=SHORT_NAMES))

# Label-only descriptions (model 5), with their length in words
if label_descriptions:
    display(pd.DataFrame([{"Product": SHORT_NAMES.get(p, p), "words": len(t.split()), "description": t}
                          for p, t in sorted(label_descriptions.items())]).set_index("Product"))
pd.set_option("display.max_colwidth", 200)

Reading both sets of descriptions:
- **Each description describes its own product accurately.** The complaint-informed set uses the consumers' own situations (escrow and loan modifications; repossession and title release; frozen balances on payment apps); the label-only set reads like a textbook definition and names related products to tell them apart.
- **Several complaint-informed descriptions share themes.** Credit card, debt collection, debt management and student loan all mention disputing information on credit reports; checking/savings, money transfer and prepaid card all describe fraud, frozen funds and refused refunds. These are the same overlaps behind the errors in 3.9.
- **Five sample complaints can pull a description off course** (the debt management description mentions student loan forgiveness), which is one reason k matters for model 6.
- **The label-only descriptions start with a Markdown heading** (for example `# Credit Card`), which adds the product name as extra words; this is harmless for embeddings but worth removing in a next version.

### 4.3 Error analysis: why did the chosen model get these wrong?

We send **every test complaint that the TF-IDF model chosen in 3.7 misclassified** to Haiku 5.5 (prompt 4), which says which product fits best, whether the complaint is ambiguous, and which clues point there. Comparing its answer with the consumer's label and our model's prediction:
- **Model error:** Claude agrees with the consumer, so our model was wrong.
- **Possible mislabel:** Claude agrees with our model, so the consumer's chosen product may be the debatable one.
- **Unclear:** Claude picks a third product.

Answers are saved by Complaint ID. Complaints already reviewed in an earlier run are reused, and only new ones are sent (a few cents).

In [ ]:
# Every test complaint the chosen model got wrong, with its prediction attached
errors = test_df[test_pred != y_test].assign(model_product=test_pred[test_pred != y_test])
ERRORS_FILE = DATA_DIR / "claude_error_analysis.csv"

# Reuse saved answers (matched by Complaint ID); send only complaints not reviewed yet, or whose request failed
saved = pd.read_csv(ERRORS_FILE) if ERRORS_FILE.exists() else pd.DataFrame(columns=["Complaint ID", "claude_clues"])
saved = saved[~saved["claude_clues"].astype(str).str.startswith("ERROR")]
to_send = errors[~errors["Complaint ID"].astype(str).isin(saved["Complaint ID"].astype(str))]
if len(to_send) and api_key():
    saved = pd.concat([saved, explain_errors(to_send, CLAUDE_MODELS["Haiku 5.5"])], ignore_index=True)
    saved.to_csv(ERRORS_FILE, index=False)
elif len(to_send):
    print(f"{len(to_send)} errors have no saved Claude review and there is no API key: they are left out.")

# Keep the reviews of the CURRENT model's errors, with its current prediction
error_results = errors[["Complaint ID", "Product", "model_product"]].astype({"Complaint ID": str}).merge(
    saved.drop(columns=["consumer_product", "model_product"], errors="ignore").astype({"Complaint ID": str}),
    on="Complaint ID").rename(columns={"Product": "consumer_product"})
print(f"Test errors: {len(errors)}   reviewed by Claude: {len(error_results)}")

if len(error_results):
    # Compare Claude's product with the consumer's label and our model's prediction
    error_results["verdict"] = error_results.apply(error_verdict, axis=1)
    verdicts = error_results["verdict"].value_counts()
    display(pd.DataFrame({"complaints": verdicts, "share": (verdicts / verdicts.sum()).round(2)}))
    print(f"Claude flagged as ambiguous (could fit more than one product): "
          f"{(error_results['claude_ambiguous'] == 'YES').mean():.0%}")
    # Where Claude sides with our model: which consumer labels look debatable?
    mislabels = error_results[error_results["verdict"] == "Possible mislabel"]
    display(mislabels.groupby(["consumer_product", "model_product"]).size()
            .sort_values(ascending=False).head(8).rename("complaints").reset_index().replace(SHORT_NAMES))

*Interpretation to be written after the full run: share of model errors, possible mislabels and unclear cases for the SVM's 397 test errors, and the main mislabel pairs. (Ketty's earlier review of logistic regression's errors found that Claude sided with our model, not the consumer, on 36% of them, mainly debt collection vs credit reporting and the payment products.)*

### 4.4 Spot-check: do we agree with Claude?

Claude's error-analysis answers are judgements, so we checked them by hand. The cell below creates `Data/claude_hand_check.csv` with **30 random rows** from the error analysis, including the full narrative. Each team member reads the narrative and fills in:
- `human_agrees`: **Y** if Claude's product is a reasonable reading of the complaint, **N** if not;
- `human_note`: a few words on why, especially for every **N**.

Then re-run the cell to see the agreement rate. If agreement is low, we revise the prompt and re-run 4.4 before relying on it. Any **N** cases go into the AI usage log as examples of Claude getting something wrong.

The file is created once and then kept, so the team's answers are never overwritten by a re-run.

In [ ]:
# File the team fills in by hand
HAND_CHECK = DATA_DIR / "claude_hand_check.csv"

# First run: draw 30 random error-analysis rows with their narratives and save them for checking
if len(error_results) and not HAND_CHECK.exists():
    to_check = error_results.sample(30, random_state=1).merge(
        test_df[["Complaint ID", TEXT_COL]].astype({"Complaint ID": str}),
        on="Complaint ID")
    to_check = to_check[["Complaint ID", "consumer_product", "model_product", "claude_product",
                         "claude_ambiguous", "claude_clues", TEXT_COL]]
    to_check["human_agrees"] = ""      # fill in Y or N
    to_check["human_note"] = ""
    to_check.to_csv(HAND_CHECK, index=False)
    print(f"Created {HAND_CHECK.name}: fill in human_agrees (Y/N) and human_note, then re-run this cell.")

# Later runs: count the Y / N answers and list the disagreements
if HAND_CHECK.exists():
    checked = pd.read_csv(HAND_CHECK)
    answered = checked["human_agrees"].fillna("").astype(str).str.strip().str.upper().isin(["Y", "N"])
    if answered.sum() == 0:
        print(f"{HAND_CHECK.name} exists but has no answers yet.")
    else:
        agree = checked.loc[answered, "human_agrees"].astype(str).str.strip().str.upper().eq("Y")
        print(f"Checked: {answered.sum()} of {len(checked)}   We agree with Claude on {agree.mean():.0%} "
              f"({agree.sum()} of {answered.sum()})")
        display(checked.loc[answered & ~agree.reindex(checked.index, fill_value=True),
                            ["consumer_product", "claude_product", "claude_clues", "human_note"]])

*Agreement rate and the cases where we disagreed with Claude: to be added once the team has filled in `Data/claude_hand_check.csv` (30 rows drawn from the error analysis).*

### 4.5 API usage, cost and speed

The spend log lists every batch sent from Ketty's runs, including the failed first zero-shot run and its retry, so its total is what was charged to that key. Rebecca's label-only descriptions (11 short requests) were generated in her own run and are not in the log. The timing table compares how long each approach takes to route the 1,469 test complaints on a laptop.

In [ ]:
# Every batch sent and charged (Ketty's spend log)
SPEND_LOG = DATA_DIR / "claude_spend_log.csv"
if SPEND_LOG.exists():
    spend = pd.read_csv(SPEND_LOG)
    display(spend)
    print(f"Total charged in the spend log: ${spend['cost_usd'].sum():.2f} of the ${BUDGET_USD:.2f} notebook budget")
    if zero_shot is not None:
        zs_cost = cost_usd(zero_shot["input_tokens"].sum(), zero_shot["output_tokens"].sum(), CLAUDE_MODELS["Haiku 5.5"])
        print(f"Zero-shot routing at Haiku 5.5 prices: ${zs_cost / len(zero_shot) * 10_000:.2f} per 10,000 complaints")
else:
    print("No spend log found in Data/.")

# Local routing time for the 1,469 test complaints (excluding training)
timing = []
for name, model in fitted.items():
    start = time.perf_counter()
    model.predict(X_test)
    timing.append({"approach": f"TF-IDF + {name}", "seconds": time.perf_counter() - start})
if sbert is not None:
    start = time.perf_counter()
    sbert.encode(X_test.tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    timing.append({"approach": "MiniLM embedding (models 5, 6)", "seconds": time.perf_counter() - start})
timing = pd.DataFrame(timing).set_index("approach")
timing["ms per complaint"] = timing["seconds"] / len(X_test) * 1000
timing.round(3)

- **Total charged in the spend log: about $1.14** of the team's $25 budget (plus a few cents for the error-analysis reviews of the SVM's new errors). Rebecca's 11 label-only descriptions are not in the log; at Haiku prices they cost well under one cent.
- **The most expensive part per call was the Sonnet descriptions** ($0.48 for 33 calls, because each prompt includes 5–20 full complaints), and they fed the weakest models.
- **Zero-shot routing with Haiku 5.5 costs about $0.72 per 10,000 complaints.** The first zero-shot run lost 2,964 answers to one setting and was paid for anyway ($0.25 estimated), which is why the log records every batch.
- **Speed:** the TF-IDF models route each complaint in about **0.13 milliseconds** on a laptop, at no per-complaint cost; the sentence-embedding models take a few milliseconds per complaint; Claude zero-shot took about 10 minutes for 4,895 complaints.

### 4.6 Trade-offs: when was Claude worth it?

| Claude use | Result | Cost | Worth it? |
|---|---|---|---|
| Zero-shot routing (Haiku) | 72.6% test accuracy, tied with the trained SVM (73.0%), with the highest precision | About $0.72 per 10,000 complaints and about 10 minutes per 5,000 | **As a second opinion:** no training data needed and very precise, but the text leaves the company, 1.8% of answers are off-list, and there is no confidence score |
| Descriptions from sample complaints (Sonnet), for models 4 and 6 | 18.3% with spaCy, 48.7% with MiniLM | $0.48 one-off | **Only for a team with no labelled data:** cheap, but far below a trained model |
| Descriptions from product names (Haiku), for model 5 | 43.5% with MiniLM | Under one cent | **Same as above**, and the most private: no complaint text is sent |
| Error analysis (Haiku) | Shows which errors are debatable labels rather than model mistakes (4.3) | A few cents | **Yes:** a few cents changed how we read our own accuracy, pending the hand check (4.4) |

## 5. Final report

### 5.1 Key findings

1. **The complaint text alone routes about three in four complaints to the right team.** The TF-IDF linear SVM, chosen by cross-validation before testing, scores **73.0%** test accuracy (macro F1 0.728) against 9.3% for the baseline. Logistic regression (73.1%) and Claude zero-shot (72.6%) are statistically tied with it.
2. **The model knows when it is unsure.** Its most confident half of complaints is routed correctly about **91%** of the time; its least confident are close to a coin flip (3.12).
3. **Performance is uneven across teams.** Mortgage and student loan reach about 88% recall; **debt management only 52%**, mostly lost to debt collection. Payment products (prepaid, money transfer, checking/savings) are confused with each other because they share apps and cards (3.9).
4. **The model reads the situation, not just the label.** Hiding every product-name word costs 5.8 points (73.0% → 67.2%), not the model's skill (3.11). But **21 of the 110 strongest terms are company names**, a shortcut that may not transfer to new companies (3.10).
5. **Our instructor's embedding approach is cheap but weak here.** With spaCy word vectors it reaches 18%; better sentence embeddings raise it to 44–49%, still far below the trained models (3.4, 3.5).
6. **Short complaints are harder:** 62.8% for the shortest quarter vs 79.2% for the longest (3.13).

### 5.2 Business implications: what the routing team does differently tomorrow

| Recommendation | Evidence | Actionable or diagnostic? |
|---|---|---|
| **Pilot the TF-IDF linear SVM as the main router.** It is the most accurate on training data, runs locally in a fraction of a millisecond per complaint, and complaint text never leaves the company | 3.7, 3.8, 4.5 | Actionable: a routing decision |
| **Auto-route only confident cases; send the rest to a person.** For example, auto-route the most confident half (about 91% accurate) and review the other half | 3.12 | Actionable: sets the review workload |
| **Ask a follow-up question when a complaint is short.** Short complaints are the least accurate; one intake question ("which product is this about?") gives the model or the agent the missing clue | 3.13 | Actionable: changes the intake form |
| **Track recall for each team, starting with debt management.** Overall accuracy hides that half of debt management's complaints go elsewhere | 3.9 | Diagnostic: a monitoring metric |
| **Re-check accuracy when new companies appear.** Company names are among the strongest clues | 3.10 | Diagnostic: triggers a review, nobody changes how consumers write |
| **Keep Claude zero-shot as a governed second opinion** for low-confidence cases, after privacy and cost approval | 3.8, 4.6 | Actionable once approved |

### 5.3 Ethics and privacy

- **Sending complaint text to an outside service.** Zero-shot classification, the complaint-informed descriptions and the error analysis send full narratives to the Claude API. CFPB redacts its published narratives (82% contain `XXXX`), but a company's own emails and chats would contain names, account numbers and addresses. Before production, the company would need a data-processing agreement, redaction before sending, and a retention policy. Rebecca's label-only descriptions show a private alternative: they send no complaint text at all.
- **Who bears the errors.** A misrouted complaint delays help for that consumer. Short complaints and possibly servicemember complaints are routed less accurately (3.13); a human-review queue for low-confidence and short complaints protects them most.
- **Labels are the consumer's choice.** The model learns what consumers picked, which is sometimes debatable (4.3); it should not be used to judge whether a consumer "chose wrong".
- **No demographic data is used.** The only group fields are the CFPB tags; we report them descriptively and do not claim the model is fair.
- **API key.** Each team member keeps the key in a personal `.env` file, listed in `.gitignore`; it is never printed or committed.

### 5.4 AI usage log

| Task | AI tool | What the AI produced | How we verified it | What went wrong or changed |
|---|---|---|---|---|
| CFPB data download | Claude (chat) | Python script for the API query | Checked row counts, date range and product counts against the CFPB site | The API's `field=all` caused a Bad Request; narratives were missing from the API, so we used the archive file |
| Notebook code (sections 1–4) | Claude (chat) | Draft code and comments | Ran every cell top to bottom; compared figures and tables with the data | A comment after `%matplotlib inline` caused an error; the lemmatizer turned "was" into "wa" and "Wells" into "well" (we kept the non-lemmatizing cleaner); a regex `\b` became a backspace character in the company-word check (fixed after the word list looked wrong); a pandas ID-type error in the zero-shot repair; `background_gradient` needed a newer matplotlib |
| Product descriptions, models 4–6 | Claude Sonnet 5.5 (sample complaints); Claude Haiku (product names only) | 33 and 11 descriptions | Read all of them (4.2); measured their effect on accuracy (3.4, 3.5) | Five sample complaints could pull a description off course |
| Zero-shot classification, model 7 | Claude Haiku 5.5 | One product per complaint | Scored against the labels; counted off-list answers | The first run lost 2,964 of 4,895 answers because Haiku "thinks" first and our code read only the first block; fixed by turning thinking off and re-sending only the failed complaints. The API also rejected the `temperature` parameter |
| Error analysis | Claude Haiku 5.5 | Product, ambiguity flag and clues for each error | 30-row hand check by the team (4.4) | Sonnet 5.5 rejected our thinking setting, so we switched to Haiku; Haiku marked 92% of errors as "ambiguous", so the flag is not informative |
| Budget control | Claude (chat) | Cost estimate and spend log | Compared estimates with the logged cost | The first estimate (4 characters per token) was slightly below the real cost; we raised the margin |
| Presentation | Claude (chat) | Slide drafts and speaker notes | Every number checked against this notebook | — |

### 5.5 Limitations and next steps

- **Balanced sample.** Real complaint traffic is about 91% credit reporting (1.3), so live accuracy will differ; a model trained on balanced data over-predicts small products.
- **One split, no significance tests.** The top three approaches are within half a point of each other; we do not claim one is better than the others on the test set.
- **No tuning.** Models keep fixed settings (3.7); tuning C and the TF-IDF thresholds is a next step.
- **Uncalibrated confidence.** SVM score gaps are not probabilities; review cut-offs need calibration and fresh validation data.
- **Description provenance.** The complaint IDs Claude read for the descriptions are fixed by the random seed, but model 5 and model 6 descriptions were made with different prompts, so their gap has no single cause.
- **Next steps:** test on the real product mix, calibrate the confidence score and set the review cut-off, remove redaction and dollar-amount tokens, tune the SVM, and re-run the hand check on a larger sample.

### 5.6 Reproducibility

Run the notebook from the repository root or the `Notebook/` folder with the packages in `requirements.txt` (`pip install -r requirements.txt` also installs spaCy's `en_core_web_md`). All Claude answers are loaded from `Data/` and `outputs/rebecca_modelling/`, so a full run needs **no API key and costs nothing**; without those files, the Claude steps are skipped with a message. The random seed (42) fixes the split, the folds and the samples. Charts and tables are saved to `outputs/`.